# Workshop: Simulación de Escenarios Críticos

Esta versión corrige los puntos solicitados para entrega académica:

- Split temporal y principal con control anti-leakage por `factura_id`.
- Selección de features sin excluir variables válidas solo por contener `corte`.
- OneHotEncoder para variables categóricas nominales.
- Drift por sector evaluando todos los `sector_*` one-hot.
- Exportación de `classification_report_baseline.csv` y `classification_report_worst_scenario.csv`.
- Informe Markdown sin placeholders y con interpretación del peor escenario `missing_dirigido_gestion`.
- Outputs centralizados en `critical_scenarios_outputs`.


In [ ]:
# ── CELDA 1 — Instalación de dependencias ──────────────────────────────────────
!pip install xgboost scikit-learn pandas numpy matplotlib joblib scipy --quiet

In [ ]:
# ── CELDA 2 — Imports y configuración global ───────────────────────────────────
import os
import sys
import json
import time
import warnings
import traceback
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import joblib

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from scipy import stats

import xgboost as xgb

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# Semilla global
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Dependencias cargadas correctamente.")
print(f"   XGBoost: {xgb.__version__}")
print(f"   scikit-learn importado OK")
print(f"   Python: {sys.version.split()[0]}")


In [ ]:
# ── CELDA 3 — Clonación del repositorio ────────────────────────────────────────
import subprocess

REPO_URL  = "https://github.com/kevinUbe23/GRUPO_3.git"
REPO_DIR  = Path("/content/GRUPO_3")

if not REPO_DIR.exists():
    print("📥 Clonando repositorio...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("❌ Error al clonar:", result.stderr)
    else:
        print("✅ Repositorio clonado exitosamente.")
else:
    print("ℹ️  Repositorio ya existe en disco.")

# Listar estructura de alto nivel
for p in sorted(REPO_DIR.iterdir()):
    print(f"  {'📁' if p.is_dir() else '📄'} {p.name}")

In [ ]:
# ── CELDA 4 — Búsqueda automática de archivos CSV relevantes ───────────────────
def buscar_csv(repo_dir: Path, nombre: str) -> list[Path]:
    """Busca recursivamente un archivo CSV por nombre (case-insensitive)."""
    resultados = []
    for p in repo_dir.rglob("*.csv"):
        if nombre.lower() in p.name.lower():
            resultados.append(p)
    return sorted(resultados)

# Prioridades de búsqueda
PRIORIDADES = [
    "features_ml_prepared",
    "features_ml",
    "train_facturas_ids",
    "test_facturas_ids",
    "model_eval",
]

rutas_encontradas = {}
for nombre in PRIORIDADES:
    rutas = buscar_csv(REPO_DIR, nombre)
    if rutas:
        rutas_encontradas[nombre] = rutas
        for r in rutas:
            print(f"✅ [{nombre}] → {r.relative_to(REPO_DIR)}")
    else:
        print(f"⚠️  [{nombre}] → NO ENCONTRADO")

# Mostrar TODOS los CSV encontrados en el repo para diagnóstico
print("\n📋 Todos los CSV en el repositorio:")
todos_csv = sorted(REPO_DIR.rglob("*.csv"))
for csv_path in todos_csv:
    size_kb = csv_path.stat().st_size / 1024
    print(f"   {csv_path.relative_to(REPO_DIR)}  [{size_kb:.1f} KB]")

In [ ]:
# ── CELDA 5 — Carga inteligente del dataset ────────────────────────────────────
def cargar_dataset_principal(rutas_encontradas: dict, repo_dir: Path) -> pd.DataFrame:
    """
    Carga el dataset principal siguiendo el orden de prioridades:
    1. features_ml_prepared.csv
    2. features_ml.csv
    3. Cualquier CSV con 'feature' en el nombre
    """
    candidatos = []

    if "features_ml_prepared" in rutas_encontradas:
        candidatos = rutas_encontradas["features_ml_prepared"]
        tipo = "prepared"
    elif "features_ml" in rutas_encontradas:
        candidatos = rutas_encontradas["features_ml"]
        tipo = "raw"
    else:
        # Búsqueda amplia
        candidatos = buscar_csv(repo_dir, "feature")
        tipo = "genérico"

    if not candidatos:
        raise FileNotFoundError(
            "❌ No se encontró ningún dataset válido. "
            "Verifica el repositorio manualmente."
        )

    # Elegir el más grande (más datos)
    ruta_elegida = max(candidatos, key=lambda p: p.stat().st_size)
    print(f"📂 Dataset seleccionado [{tipo}]: {ruta_elegida.relative_to(repo_dir)}")

    df = pd.read_csv(ruta_elegida, low_memory=False)
    print(f"   Shape: {df.shape}")
    print(f"   Columnas: {list(df.columns[:10])} {'...' if len(df.columns) > 10 else ''}")
    return df, ruta_elegida, tipo

df_original, ruta_dataset, tipo_dataset = cargar_dataset_principal(
    rutas_encontradas, REPO_DIR
)
print(f"\n✅ Dataset cargado: {df_original.shape[0]:,} filas × {df_original.shape[1]} columnas")

In [ ]:
# ── CELDA 6 — Validación y diagnóstico del dataset ────────────────────────────
print("=" * 65)
print("  DIAGNÓSTICO DEL DATASET")
print("=" * 65)

print(f"\n📊 Shape: {df_original.shape}")
print(f"\n🏷️  Tipos de datos:\n{df_original.dtypes.value_counts().to_string()}")

print(f"\n🔢 Valores nulos por columna (con nulos):")
nulos = df_original.isnull().sum()
nulos_cols = nulos[nulos > 0].sort_values(ascending=False)
if len(nulos_cols):
    print(nulos_cols.to_string())
else:
    print("   ✅ Sin valores nulos en el dataset original")

print(f"\n🎯 Búsqueda de columna TARGET:")
candidatos_target = [
    c for c in df_original.columns
    if any(k in c.lower() for k in [
        "target", "target_mora", "estado_mora", "bucket_mora",
        "clase", "label", "status"
    ])
]
# También considerar columnas que contengan clases esperadas.
for c in df_original.columns:
    if c not in candidatos_target:
        vals = set(df_original[c].dropna().astype(str).unique()[:20])
        if vals & {"on_time", "+30", "+60", "+90"}:
            candidatos_target.append(c)

print(f"   Candidatos: {candidatos_target}")

TARGET_COL = None
clases_esperadas = {"on_time", "+30", "+60", "+90"}
for candidato in candidatos_target:
    valores_unicos = set(df_original[candidato].dropna().astype(str).unique())
    if valores_unicos & clases_esperadas:
        TARGET_COL = candidato
        print(f"   ✅ TARGET identificado: '{TARGET_COL}'")
        print(f"   Distribución:\n{df_original[TARGET_COL].value_counts().to_string()}")
        break

if TARGET_COL is None and candidatos_target:
    TARGET_COL = candidatos_target[0]
    print(f"   ⚠️  Usando candidato por defecto: '{TARGET_COL}'")
    print(f"   Distribución:\n{df_original[TARGET_COL].value_counts().to_string()}")

if TARGET_COL is None:
    TARGET_COL = df_original.columns[-1]
    print(f"   ⚠️  Sin candidato claro. Usando última columna: '{TARGET_COL}'")

print(f"\n🔑 Búsqueda de factura_id y cliente_id:")
FACTURA_ID_COL = next(
    (c for c in df_original.columns
     if c.lower() in ["factura_id", "id_factura", "invoice_id", "factura"]),
    None
)
if FACTURA_ID_COL is None:
    FACTURA_ID_COL = next(
        (c for c in df_original.columns
         if "factura" in c.lower() and "id" in c.lower()),
        None
    )

CLIENTE_ID_COL = next(
    (c for c in df_original.columns
     if c.lower() in ["cliente_id", "id_cliente", "customer_id", "cliente"]),
    None
)
if CLIENTE_ID_COL is None:
    CLIENTE_ID_COL = next(
        (c for c in df_original.columns
         if "cliente" in c.lower() and "id" in c.lower()),
        None
    )

print(f"   factura_id_col: {FACTURA_ID_COL}")
print(f"   cliente_id_col : {CLIENTE_ID_COL}")

if FACTURA_ID_COL is None:
    raise ValueError("❌ No se encontró factura_id. Para evitar leakage, este notebook requiere partición por factura.")

print(f"\n📌 Registros únicos:")
print(f"   Facturas únicas: {df_original[FACTURA_ID_COL].nunique():,}")
if CLIENTE_ID_COL:
    print(f"   Clientes únicos : {df_original[CLIENTE_ID_COL].nunique():,}")


In [ ]:
# ── CELDA 7 — Preparación del dataset, feature engineering seguro y split ──────
def detectar_columnas_fecha(df: pd.DataFrame) -> list[str]:
    """Detecta columnas de fecha reales. No excluye variables derivadas como dias_transcurridos_corte."""
    fecha_cols = []
    for c in df.columns:
        cl = c.lower()
        if (
            cl.startswith("fecha")
            or cl.endswith("_fecha")
            or cl.endswith("_date")
            or cl in {"date", "fecha_corte", "fecha_emision", "fecha_vencimiento", "fecha_pago"}
        ):
            fecha_cols.append(c)
    return fecha_cols


def aplicar_feature_engineering_seguro(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crea variables derivadas válidas con información disponible al corte.
    Importante: estas variables NO se eliminan por contener la palabra 'corte'.
    """
    df = df.copy()

    fecha_corte_col = next((c for c in df.columns if c.lower() == "fecha_corte"), None)
    fecha_emision_col = next((c for c in df.columns if c.lower() == "fecha_emision"), None)

    if fecha_corte_col:
        df[fecha_corte_col] = pd.to_datetime(df[fecha_corte_col], errors="coerce")
    if fecha_emision_col:
        df[fecha_emision_col] = pd.to_datetime(df[fecha_emision_col], errors="coerce")

    # Feature válida: tiempo transcurrido desde emisión hasta el corte de scoring.
    if (
        "dias_transcurridos_corte" not in df.columns
        and fecha_corte_col
        and fecha_emision_col
    ):
        df["dias_transcurridos_corte"] = (
            df[fecha_corte_col] - df[fecha_emision_col]
        ).dt.days.clip(lower=0)

    # Feature válida: indica si al corte la factura ya estaba vencida.
    if "esta_vencida_al_corte" not in df.columns:
        if "dias_hasta_vence" in df.columns:
            df["esta_vencida_al_corte"] = (pd.to_numeric(df["dias_hasta_vence"], errors="coerce") <= 0).astype("int64")
        elif "dias_transcurridos_corte" in df.columns and "condicion_dias" in df.columns:
            df["esta_vencida_al_corte"] = (
                pd.to_numeric(df["dias_transcurridos_corte"], errors="coerce")
                > pd.to_numeric(df["condicion_dias"], errors="coerce")
            ).astype("int64")

    # Feature válida: mora observable al corte, no mora final.
    if "dias_mora_observable" not in df.columns:
        if "dias_hasta_vence" in df.columns:
            df["dias_mora_observable"] = (-pd.to_numeric(df["dias_hasta_vence"], errors="coerce")).clip(lower=0)
        elif "dias_transcurridos_corte" in df.columns and "condicion_dias" in df.columns:
            df["dias_mora_observable"] = (
                pd.to_numeric(df["dias_transcurridos_corte"], errors="coerce")
                - pd.to_numeric(df["condicion_dias"], errors="coerce")
            ).clip(lower=0)

    if "dias_hasta_vence_pos" not in df.columns and "dias_hasta_vence" in df.columns:
        df["dias_hasta_vence_pos"] = pd.to_numeric(df["dias_hasta_vence"], errors="coerce").clip(lower=0)

    print("✅ Feature engineering seguro aplicado.")
    for c in ["dias_transcurridos_corte", "esta_vencida_al_corte", "dias_mora_observable", "dias_hasta_vence_pos"]:
        if c in df.columns:
            print(f"   Conservada/creada: {c}")
    return df


def seleccionar_features_sin_leakage(
    df: pd.DataFrame,
    target_col: str,
    factura_id_col: str,
    cliente_id_col: str | None,
) -> tuple[list[str], list[str], list[str]]:
    """
    Selecciona features evitando leakage explícito.
    Corrección clave:
    - NO excluye automáticamente variables solo por contener 'corte'.
    - Mantiene dias_transcurridos_corte y esta_vencida_al_corte si existen.
    - Excluye target, identificadores, fechas reales y variables de desenlace/fuga explícita.
    """
    fecha_cols = detectar_columnas_fecha(df)

    excluir = {target_col, "__target_encoded__"}
    if factura_id_col:
        excluir.add(factura_id_col)
    if cliente_id_col:
        excluir.add(cliente_id_col)

    # Fechas reales: se excluyen como predictores directos, pero sus derivadas válidas se mantienen.
    excluir.update(fecha_cols)

    leakage_exact = {
        "fecha_pago", "fecha_pago_real", "fecha_cancelacion", "fecha_cobro",
        "dias_retraso_final", "dias_mora_final", "mora_final",
        "estado_pago_final", "estado_final", "resultado_final",
        "pago_realizado", "pagado", "target_encoded", "target_mora",
    }
    leakage_patterns = [
        "post_corte", "posterior_al_corte", "despues_del_corte",
        "future_", "_future", "label_", "_label", "leak",
    ]

    leakage_cols = []
    for c in df.columns:
        cl = c.lower()
        if c in excluir:
            continue
        if cl in leakage_exact:
            leakage_cols.append(c)
        elif any(p in cl for p in leakage_patterns):
            leakage_cols.append(c)

    excluir.update(leakage_cols)

    feature_cols = [c for c in df.columns if c not in excluir]

    # Garantía explícita solicitada: conservar derivadas válidas con 'corte'.
    for c in ["dias_transcurridos_corte", "esta_vencida_al_corte"]:
        if c in df.columns and c not in feature_cols and c not in excluir:
            feature_cols.append(c)

    print(f"\n📌 Features seleccionadas: {len(feature_cols)}")
    print(f"   Fechas reales excluidas: {fecha_cols}")
    print(f"   Leakage explícito excluido: {leakage_cols}")
    print(f"   Identificadores excluidos: {[x for x in [factura_id_col, cliente_id_col] if x]}")
    print("   ✅ No se excluyen variables solo por contener la palabra 'corte'.")

    conservadas_corte = [c for c in feature_cols if "corte" in c.lower()]
    if conservadas_corte:
        print(f"   Variables con 'corte' conservadas: {conservadas_corte}")

    return feature_cols, fecha_cols, leakage_cols


def normalizar_nominales(df: pd.DataFrame, feature_cols: list[str]) -> tuple[pd.DataFrame, list[str]]:
    """
    Fuerza variables nominales codificadas como enteros a categoría, para que luego usen OneHotEncoder.
    """
    df = df.copy()
    nominales = []
    candidatos = [
        "ultimo_resultado_enc", "sector_enc", "sector", "industria", "rubro", "segmento"
    ]
    for c in candidatos:
        if c in feature_cols and c in df.columns:
            df[c] = df[c].astype("object")
            nominales.append(c)
    if nominales:
        print(f"✅ Variables nominales tratadas como categóricas: {nominales}")
    return df, nominales


def verificar_no_interseccion_facturas(df_train, df_test, factura_id_col: str, contexto: str) -> None:
    train_ids = set(df_train[factura_id_col].dropna().astype(str))
    test_ids = set(df_test[factura_id_col].dropna().astype(str))
    inter = train_ids & test_ids
    print(f"\n🔒 Control anti-leakage ({contexto}):")
    print(f"   Facturas train: {len(train_ids):,}")
    print(f"   Facturas test : {len(test_ids):,}")
    print(f"   Intersección  : {len(inter):,}")
    assert len(inter) == 0, f"❌ Leakage detectado: {len(inter)} factura_id aparecen en train y test."


def preparar_dataset(df: pd.DataFrame,
                     target_col: str,
                     factura_id_col: str,
                     cliente_id_col: str | None,
                     tipo_dataset: str,
                     repo_dir: Path,
                     random_state: int = 42) -> tuple:
    """
    Prepara el dataset con control metodológico:
    - Limpia target.
    - Crea variables derivadas válidas al corte.
    - Codifica el target.
    - Separa train/test por factura_id.
    - Garantiza que no exista intersección de factura_id entre train y test.
    - Selecciona features sin excluir automáticamente variables por contener 'corte'.
    """
    df = df.copy()
    df = aplicar_feature_engineering_seguro(df)

    # Limpiar target.
    df = df.dropna(subset=[target_col])
    df[target_col] = df[target_col].astype(str).str.strip()

    # Codificar target.
    le = LabelEncoder()
    df["__target_encoded__"] = le.fit_transform(df[target_col])
    print(f"\n✅ Clases del target: {list(le.classes_)}")
    print(f"   Encodings: {dict(zip(le.classes_, le.transform(le.classes_)))}")

    # Partición predefinida si existe.
    train_ids_paths = buscar_csv(repo_dir, "train_facturas_ids")
    test_ids_paths  = buscar_csv(repo_dir, "test_facturas_ids")

    usar_ids_predef = (
        factura_id_col is not None
        and len(train_ids_paths) > 0
        and len(test_ids_paths) > 0
    )

    if usar_ids_predef:
        print(f"\n🔀 Usando partición predefinida por factura_id...")
        train_ids = pd.read_csv(train_ids_paths[0]).iloc[:, 0].astype(str).values
        test_ids  = pd.read_csv(test_ids_paths[0]).iloc[:, 0].astype(str).values
        inter_ids = set(train_ids) & set(test_ids)
        assert len(inter_ids) == 0, f"❌ Leakage en archivos de IDs: {len(inter_ids)} factura_id repetidos."

        ids_str = df[factura_id_col].astype(str)
        df_train  = df[ids_str.isin(train_ids)].copy()
        df_test   = df[ids_str.isin(test_ids)].copy()
        print(f"   Train: {len(df_train):,} | Test: {len(df_test):,}")

    else:
        print(f"\n🔀 Split por grupos de factura_id (evita leakage)...")
        grupos = df[factura_id_col].astype(str).values
        gss = GroupShuffleSplit(
            n_splits=1, test_size=0.2, random_state=random_state
        )
        train_idx, test_idx = next(gss.split(df, groups=grupos))
        df_train = df.iloc[train_idx].copy()
        df_test  = df.iloc[test_idx].copy()
        print(f"   Train: {len(df_train):,} | Test: {len(df_test):,}")

    verificar_no_interseccion_facturas(df_train, df_test, factura_id_col, "split principal")

    # Features sin leakage.
    feature_cols, fecha_cols, leakage_cols = seleccionar_features_sin_leakage(
        df, target_col, factura_id_col, cliente_id_col
    )

    df, nominal_cols = normalizar_nominales(df, feature_cols)
    df_train, _ = normalizar_nominales(df_train, feature_cols)
    df_test, _ = normalizar_nominales(df_test, feature_cols)

    X_train = df_train[feature_cols].copy()
    X_test  = df_test[feature_cols].copy()
    y_train = df_train["__target_encoded__"].values
    y_test  = df_test["__target_encoded__"].values

    split_audit = {
        "n_registros_train": len(df_train),
        "n_registros_test": len(df_test),
        "n_facturas_train": df_train[factura_id_col].nunique(),
        "n_facturas_test": df_test[factura_id_col].nunique(),
        "interseccion_factura_id": 0,
        "feature_cols": feature_cols,
        "fecha_cols_excluidas": fecha_cols,
        "leakage_cols_excluidas": leakage_cols,
        "nominal_cols_onehot": nominal_cols,
    }

    return X_train, X_test, y_train, y_test, le, feature_cols, fecha_cols, leakage_cols, nominal_cols, split_audit, df

X_train, X_test, y_train, y_test, le, feature_cols, fecha_cols, leakage_cols, nominal_cols, split_audit, df_model = preparar_dataset(
    df_original, TARGET_COL, FACTURA_ID_COL, CLIENTE_ID_COL, tipo_dataset, REPO_DIR, RANDOM_STATE
)

print(f"\n✅ Split completado:")
print(f"   X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"   Clases en y_test: {np.unique(y_test)} → {le.classes_[np.unique(y_test)]}")


In [ ]:
# ── CELDA 8 — Construcción del pipeline de preprocesamiento ────────────────────
def crear_one_hot_encoder():
    """Compatibilidad entre versiones de scikit-learn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def construir_pipeline_preprocesamiento(X: pd.DataFrame) -> ColumnTransformer:
    """
    Construye un ColumnTransformer que:
    - Imputa y escala variables numéricas.
    - Imputa y codifica variables nominales/categóricas con OneHotEncoder(handle_unknown="ignore").
    Corrección aplicada: se elimina OrdinalEncoder para variables nominales.
    """
    num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    print(f"   Numéricas: {len(num_cols)} columnas")
    print(f"   Categóricas nominales: {len(cat_cols)} columnas")
    if cat_cols:
        print(f"   OneHotEncoder aplicado a: {cat_cols}")

    transformers = []

    if num_cols:
        num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
        ])
        transformers.append(("num", num_pipeline, num_cols))

    if cat_cols:
        cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", crear_one_hot_encoder()),
        ])
        transformers.append(("cat", cat_pipeline, cat_cols))

    ct = ColumnTransformer(transformers=transformers, remainder="drop")
    return ct, num_cols, cat_cols

print("🔧 Construyendo pipeline de preprocesamiento...")
preprocessor, num_cols, cat_cols = construir_pipeline_preprocesamiento(X_train)

# Pipeline completo: preprocesador + XGBoost.
n_clases = len(le.classes_)
xgb_model = xgb.XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    eval_metric       = "mlogloss",
    num_class         = n_clases if n_clases > 2 else None,
    objective         = "multi:softprob" if n_clases > 2 else "binary:logistic",
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbosity         = 0,
)

pipeline_completo = Pipeline([
    ("preprocessor", preprocessor),
    ("model",        xgb_model),
])

print("✅ Pipeline construido con OneHotEncoder para categóricas nominales.")


In [ ]:
# ── CELDA 9 — Entrenamiento o carga del modelo ─────────────────────────────────
OUTPUT_DIR = Path("/content/critical_scenarios_outputs")
ARTIFACTS_DIR  = OUTPUT_DIR / "artifacts"
TABLES_DIR     = OUTPUT_DIR / "tables"
FIGURES_DIR    = OUTPUT_DIR / "figures"
REPORT_DIR     = OUTPUT_DIR / "report"

for d in [ARTIFACTS_DIR, TABLES_DIR, FIGURES_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_PATH        = ARTIFACTS_DIR / "model_baseline.joblib"
PREPROC_PATH      = ARTIFACTS_DIR / "preprocessing_pipeline.joblib"
SCENARIO_CFG_PATH = ARTIFACTS_DIR / "scenario_config.json"

# Se fuerza reentrenamiento para evitar cargar artefactos viejos con OrdinalEncoder
# o con selección de variables previa a esta corrección.
FORCE_RETRAIN = True

def entrenar_modelo(pipeline, X_train, y_train):
    print("🏋️  Entrenando modelo XGBoost...")
    t0 = time.perf_counter()
    pipeline.fit(X_train, y_train)
    t_total = time.perf_counter() - t0
    print(f"   ✅ Entrenamiento completado en {t_total:.1f}s")
    return pipeline

if MODEL_PATH.exists() and not FORCE_RETRAIN:
    print("📦 Cargando modelo pre-entrenado...")
    pipeline_completo = joblib.load(MODEL_PATH)
    print("   ✅ Modelo cargado.")
else:
    pipeline_completo = entrenar_modelo(pipeline_completo, X_train, y_train)
    joblib.dump(pipeline_completo, MODEL_PATH)
    joblib.dump(pipeline_completo.named_steps["preprocessor"], PREPROC_PATH)
    print(f"   💾 Modelo guardado en {MODEL_PATH}")

# Config de escenarios y auditoría del split.
scenario_config = {
    "version": "final",
    "random_state": RANDOM_STATE,
    "target_col": TARGET_COL,
    "factura_id_col": FACTURA_ID_COL,
    "cliente_id_col": CLIENTE_ID_COL,
    "clases": list(le.classes_),
    "n_clases": n_clases,
    "num_cols": num_cols,
    "cat_cols_onehot": cat_cols,
    "feature_cols": feature_cols,
    "fecha_cols_excluidas": fecha_cols,
    "leakage_cols_excluidas": leakage_cols,
    "nominal_cols_onehot": nominal_cols,
    "split_audit": split_audit,
    "tipo_dataset": tipo_dataset,
    "dataset_path": str(ruta_dataset),
}
with open(SCENARIO_CFG_PATH, "w", encoding="utf-8") as f:
    json.dump(scenario_config, f, indent=2, default=str)

print(f"   💾 Config guardada en {SCENARIO_CFG_PATH}")


In [ ]:
# ── CELDA 10 — Función central de evaluación ───────────────────────────────────
def evaluate_model(pipeline, X: pd.DataFrame, y_true: np.ndarray,
                   le: LabelEncoder, scenario_name: str = "baseline",
                   medir_latencia: bool = True) -> dict:
    """
    Evalúa el pipeline sobre (X, y_true) y devuelve un dict con todas las métricas.
    """
    clases     = le.classes_
    clases_idx = list(range(len(clases)))

    # ── Medir latencia ──
    t0    = time.perf_counter()
    y_pred = pipeline.predict(X)
    t_tot  = time.perf_counter() - t0
    lat_por_registro = (t_tot / len(X)) * 1_000  # ms

    # ── Métricas globales ──
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1_mac  = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1_wei  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    prec_mac= precision_score(y_true, y_pred, average="macro",    zero_division=0)
    rec_mac = recall_score(y_true, y_pred, average="macro",    zero_division=0)

    # ── Métricas por clase ──
    f1_por_clase   = f1_score(y_true, y_pred, average=None,
                               labels=clases_idx, zero_division=0)
    rec_por_clase  = recall_score(y_true, y_pred, average=None,
                                   labels=clases_idx, zero_division=0)
    prec_por_clase = precision_score(y_true, y_pred, average=None,
                                      labels=clases_idx, zero_division=0)

    # ── Localizar clase +90 ──
    clase_90_nombre  = "+90"
    clase_90_idx     = None
    for i, c in enumerate(clases):
        if "+90" in str(c) or "90" in str(c):
            clase_90_idx = i
            clase_90_nombre = str(c)
            break

    recall_90 = rec_por_clase[clase_90_idx] if clase_90_idx is not None else np.nan
    f1_90     = f1_por_clase[clase_90_idx]  if clase_90_idx is not None else np.nan

    # ── Predicciones inválidas ──
    n_invalidas = int(np.sum((y_pred < 0) | (y_pred >= len(clases))))

    resultados = {
        "escenario":               scenario_name,
        "accuracy":                round(acc,      4),
        "balanced_accuracy":       round(bal_acc,  4),
        "f1_macro":                round(f1_mac,   4),
        "f1_weighted":             round(f1_wei,   4),
        "precision_macro":         round(prec_mac, 4),
        "recall_macro":            round(rec_mac,  4),
        f"recall_{clase_90_nombre}": round(recall_90, 4) if not np.isnan(recall_90) else np.nan,
        f"f1_{clase_90_nombre}":     round(f1_90,     4) if not np.isnan(f1_90)     else np.nan,
        "latencia_total_s":        round(t_tot,           4),
        "latencia_ms_por_reg":     round(lat_por_registro, 4),
        "n_registros":             len(X),
        "n_invalidas":             n_invalidas,
        "degradacion_f1_macro_pct": np.nan,  # se calcula vs baseline después
    }

    # Añadir F1 por clase
    for i, clase in enumerate(clases):
        resultados[f"f1_{clase}"]  = round(f1_por_clase[i],  4)
        resultados[f"rec_{clase}"] = round(rec_por_clase[i], 4)

    return resultados, y_pred

print("✅ Función evaluate_model definida.")

In [ ]:
# ── CELDA 11 — BASELINE: evaluación sobre test limpio ──────────────────────────
print("=" * 65)
print("  EVALUACIÓN BASELINE")
print("=" * 65)

def guardar_classification_report_csv(y_true, y_pred, le, path_out: Path) -> pd.DataFrame:
    """Exporta classification_report en formato CSV académico y reproducible."""
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(le.classes_))),
        target_names=list(le.classes_),
        output_dict=True,
        zero_division=0,
    )
    df_report = pd.DataFrame(report_dict).T
    df_report.to_csv(path_out, index=True)
    print(f"   💾 Classification report guardado: {path_out.name}")
    return df_report

metricas_baseline, y_pred_baseline = evaluate_model(
    pipeline_completo, X_test, y_test, le,
    scenario_name="baseline_clean"
)

print(f"\n📊 MÉTRICAS BASELINE:")
for k, v in metricas_baseline.items():
    if k not in ["escenario", "n_registros", "n_invalidas", "degradacion_f1_macro_pct"]:
        print(f"   {k:35s}: {v}")

# Guardar baseline.
df_baseline = pd.DataFrame([metricas_baseline])
df_baseline.to_csv(TABLES_DIR / "baseline_metrics.csv", index=False)
print(f"\n💾 Baseline guardado en tables/baseline_metrics.csv")

# Export solicitado: classification_report_baseline.csv.
df_report_baseline = guardar_classification_report_csv(
    y_test, y_pred_baseline, le,
    TABLES_DIR / "classification_report_baseline.csv"
)

# Matriz de confusión baseline.
def plot_confusion_matrix(y_true, y_pred, le, titulo, path_out):
    cm   = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    clases = le.classes_
    ax.set_xticks(range(len(clases))); ax.set_yticks(range(len(clases)))
    ax.set_xticklabels(clases, rotation=30, ha="right")
    ax.set_yticklabels(clases)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
    ax.set_title(titulo, pad=12)
    for i in range(len(clases)):
        for j in range(len(clases)):
            ax.text(j, i, str(cm[i, j]),
                    ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() * 0.5 else "black",
                    fontsize=12, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(path_out, bbox_inches="tight")
    plt.show()
    print(f"   💾 Guardado: {path_out.name}")
    return cm

cm_baseline = plot_confusion_matrix(
    y_test, y_pred_baseline, le,
    "Matriz de Confusión — Baseline (Test Limpio)",
    FIGURES_DIR / "confusion_matrix_baseline.png"
)

# Distribución real vs predicha.
print(f"\n📊 Distribución real en test:")
for i, c in enumerate(le.classes_):
    n = np.sum(y_test == i)
    print(f"   {c}: {n:5d} ({100*n/len(y_test):.1f}%)")
print(f"\n📊 Distribución predicha en test:")
for i, c in enumerate(le.classes_):
    n = np.sum(y_pred_baseline == i)
    print(f"   {c}: {n:5d} ({100*n/len(y_pred_baseline):.1f}%)")


In [ ]:
# ── CELDA 12 — Funciones para escenarios de perturbación ───────────────────────

# ── Variables sugeridas por el negocio ──
VARS_NEGOCIO_NUM = [
    "monto", "ratio_monto", "mora_promedio_hist", "mora_ultimo_tramo",
    "tasa_cumplimiento", "tasa_contacto_cliente", "tasa_cumpl_promesas",
    "num_gestiones_factura", "num_promesas_rotas", "promesas_total",
    "dias_hasta_vence", "dias_mora_observable",
]

def obtener_vars_numericas_disponibles(X: pd.DataFrame,
                                       vars_sugeridas: list) -> list:
    """Filtra las variables sugeridas que realmente existen en X."""
    disponibles = [v for v in vars_sugeridas if v in X.columns]
    if not disponibles:
        # Fallback: todas las numéricas
        disponibles = X.select_dtypes(include="number").columns.tolist()
    print(f"   Variables numéricas disponibles para perturbación: {disponibles}")
    return disponibles

# ─────────────────────────────────────────────────────────────
# A. RUIDO GAUSSIANO
# ─────────────────────────────────────────────────────────────
def agregar_ruido_gaussiano(X: pd.DataFrame, nivel: float,
                             vars_num: list,
                             random_state: int = 42) -> pd.DataFrame:
    """
    Agrega ruido gaussiano proporcional a std de cada variable.
    nivel = fracción de la std (ej. 0.05 = 5%).
    No altera identificadores ni target.
    """
    rng   = np.random.RandomState(random_state)
    X_mod = X.copy()
    for col in vars_num:
        if col in X_mod.columns:
            std_col = X_mod[col].std(skipna=True)
            ruido   = rng.normal(0, nivel * std_col, size=len(X_mod))
            X_mod[col] = X_mod[col] + ruido
    return X_mod

# ─────────────────────────────────────────────────────────────
# B. MISSING VALUES ALEATORIOS
# ─────────────────────────────────────────────────────────────
def agregar_missing_aleatorio(X: pd.DataFrame, fraccion: float,
                               vars_target: list = None,
                               random_state: int = 42) -> pd.DataFrame:
    """
    Introduce NaN aleatorios en fraccion de celdas de vars_target.
    Si vars_target es None, aplica sobre todas las columnas predictoras.
    """
    rng   = np.random.RandomState(random_state)
    X_mod = X.copy()
    cols  = vars_target if vars_target else X_mod.columns.tolist()
    for col in cols:
        if col in X_mod.columns:
            n_missing = int(len(X_mod) * fraccion)
            idx_missing = rng.choice(len(X_mod), size=n_missing, replace=False)
            X_mod.iloc[idx_missing, X_mod.columns.get_loc(col)] = np.nan
    return X_mod

# ─────────────────────────────────────────────────────────────
# C. MISSING VALUES DIRIGIDOS (fallo operativo)
# ─────────────────────────────────────────────────────────────
GRUPOS_MISSING_DIRIGIDO = {
    "gestion": [
        "dias_desde_ultima_gestion", "ultimo_resultado_enc",
        "tasa_contacto_cliente", "num_gestiones_factura"
    ],
    "promesas": [
        "tasa_cumpl_promesas", "num_promesas_rotas",
        "promesas_total"
    ],
    "historico": [
        "mora_promedio_hist", "mora_ultimo_tramo",
        "tasa_cumplimiento"
    ],
}

def missing_dirigido(X: pd.DataFrame, grupo: str) -> pd.DataFrame:
    """Simula fallo operativo eliminando un grupo completo de variables."""
    cols = GRUPOS_MISSING_DIRIGIDO.get(grupo, [])
    cols_presentes = [c for c in cols if c in X.columns]
    X_mod = X.copy()
    for col in cols_presentes:
        X_mod[col] = np.nan
    print(f"   Grupo '{grupo}': {len(cols_presentes)} variables anuladas "
          f"({cols_presentes})")
    return X_mod

# ─────────────────────────────────────────────────────────────
# D. OUTLIERS ARTIFICIALES
# ─────────────────────────────────────────────────────────────
def agregar_outliers(X: pd.DataFrame, factor: float,
                     col_principal: str = "monto",
                     fraccion: float = 0.05,
                     random_state: int = 42) -> pd.DataFrame:
    """
    Multiplica `col_principal` por `factor` en `fraccion` del dataset.
    """
    rng   = np.random.RandomState(random_state)
    X_mod = X.copy()
    n_mod = max(1, int(len(X_mod) * fraccion))
    idx_mod = rng.choice(len(X_mod), size=n_mod, replace=False)
    col = col_principal if col_principal in X_mod.columns else None
    if col is None:
        # Buscar columna de monto
        for c in ["monto", "amount", "valor", "importe"]:
            if c in X_mod.columns:
                col = c
                break
    if col:
        X_mod.iloc[idx_mod, X_mod.columns.get_loc(col)] *= factor
        print(f"   Outliers en '{col}': factor {factor}x en {n_mod} registros ({fraccion*100:.0f}%)")
    return X_mod

def outlier_critico_combinado(X: pd.DataFrame,
                               fraccion: float = 0.05,
                               random_state: int = 42) -> pd.DataFrame:
    """
    Crea perfiles críticos: monto alto + muchas promesas rotas +
    baja tasa de contacto + historial de mora alto.
    """
    rng   = np.random.RandomState(random_state)
    X_mod = X.copy()
    n_mod = max(1, int(len(X_mod) * fraccion))
    idx_mod = rng.choice(len(X_mod), size=n_mod, replace=False)

    cambios = {
        "monto":                lambda col: X_mod[col].quantile(0.99) * 3,
        "num_promesas_rotas":   lambda col: X_mod[col].max() * 2 + 5,
        "tasa_contacto_cliente":lambda col: 0.0,
        "mora_promedio_hist":   lambda col: X_mod[col].quantile(0.99) * 2,
        "tasa_cumplimiento":    lambda col: 0.05,
    }
    aplicados = []
    for col, fn in cambios.items():
        if col in X_mod.columns:
            X_mod.iloc[idx_mod, X_mod.columns.get_loc(col)] = fn(col)
            aplicados.append(col)
    print(f"   Outlier crítico combinado: {len(aplicados)} variables en {n_mod} registros")
    return X_mod

print("✅ Funciones de perturbación definidas.")

In [ ]:
# ── CELDA 13 — Función auxiliar: calcular degradación ─────────────────────────
def calcular_degradacion(metricas_base: dict, metricas_esc: dict,
                          metrica_key: str = "f1_macro") -> float:
    """Fórmula: ((base - esc) / base) * 100"""
    base = metricas_base.get(metrica_key, np.nan)
    esc  = metricas_esc.get(metrica_key, np.nan)
    if base and base != 0 and not np.isnan(base) and not np.isnan(esc):
        return round(((base - esc) / base) * 100, 2)
    return np.nan

def enriquecer_con_degradacion(metricas: dict, baseline: dict) -> dict:
    """Añade % de degradación para métricas principales vs baseline."""
    m = metricas.copy()
    for key in ["f1_macro", "balanced_accuracy", "recall_macro", "accuracy"]:
        base_val = baseline.get(key, np.nan)
        esc_val  = m.get(key, np.nan)
        if not (np.isnan(base_val) or np.isnan(esc_val) or base_val == 0):
            m[f"degradacion_{key}_pct"] = round(
                ((base_val - esc_val) / base_val) * 100, 2
            )
        else:
            m[f"degradacion_{key}_pct"] = np.nan
    return m

print("✅ Funciones de degradación definidas.")

In [ ]:
# ── CELDA 14 — ESCENARIOS DE RUIDO GAUSSIANO ───────────────────────────────────
print("=" * 65)
print("  SECCIÓN 1: RUIDO GAUSSIANO")
print("=" * 65)

vars_num_disponibles = obtener_vars_numericas_disponibles(X_test, VARS_NEGOCIO_NUM)
niveles_ruido = {"ruido_bajo_5pct": 0.05, "ruido_medio_10pct": 0.10, "ruido_alto_20pct": 0.20}
resultados_ruido = []

for nombre, nivel in niveles_ruido.items():
    print(f"\n🔊 Escenario: {nombre} (nivel={nivel*100:.0f}%)")
    X_ruidoso = agregar_ruido_gaussiano(X_test, nivel, vars_num_disponibles, RANDOM_STATE)
    metricas, _ = evaluate_model(
        pipeline_completo, X_ruidoso, y_test, le, scenario_name=nombre
    )
    metricas = enriquecer_con_degradacion(metricas, metricas_baseline)
    resultados_ruido.append(metricas)
    print(f"   F1-macro: {metricas['f1_macro']:.4f} | "
          f"Degradación: {metricas['degradacion_f1_macro_pct']:.2f}%")

df_ruido = pd.DataFrame(resultados_ruido)
print(f"\n📊 Resumen Ruido Gaussiano:")
print(df_ruido[["escenario", "f1_macro", "balanced_accuracy",
                "degradacion_f1_macro_pct"]].to_string(index=False))

In [ ]:
# ── CELDA 15 — ESCENARIOS DE MISSING VALUES ────────────────────────────────────
print("=" * 65)
print("  SECCIÓN 2: MISSING VALUES")
print("=" * 65)

resultados_missing = []

# 2A. Missing aleatorio por fracción
niveles_missing = {
    "missing_bajo_5pct":  0.05,
    "missing_medio_15pct": 0.15,
    "missing_alto_30pct":  0.30,
}
print("\n🔸 Missing Values Aleatorios:")
for nombre, fraccion in niveles_missing.items():
    print(f"\n   Escenario: {nombre} (fraccion={fraccion*100:.0f}%)")
    X_miss = agregar_missing_aleatorio(X_test, fraccion,
                                        vars_target=vars_num_disponibles,
                                        random_state=RANDOM_STATE)
    metricas, _ = evaluate_model(
        pipeline_completo, X_miss, y_test, le, scenario_name=nombre
    )
    metricas = enriquecer_con_degradacion(metricas, metricas_baseline)
    resultados_missing.append(metricas)
    print(f"   F1-macro: {metricas['f1_macro']:.4f} | "
          f"Degradación: {metricas['degradacion_f1_macro_pct']:.2f}%")

# 2B. Missing dirigido por grupo
print("\n🔸 Missing Values Dirigidos (fallos operativos):")
for grupo in ["gestion", "promesas", "historico"]:
    nombre_esc = f"missing_dirigido_{grupo}"
    print(f"\n   Escenario: {nombre_esc}")
    X_miss_dir = missing_dirigido(X_test, grupo)
    metricas, _ = evaluate_model(
        pipeline_completo, X_miss_dir, y_test, le, scenario_name=nombre_esc
    )
    metricas = enriquecer_con_degradacion(metricas, metricas_baseline)
    resultados_missing.append(metricas)
    print(f"   F1-macro: {metricas['f1_macro']:.4f} | "
          f"Degradación: {metricas['degradacion_f1_macro_pct']:.2f}%")

df_missing = pd.DataFrame(resultados_missing)
print(f"\n📊 Resumen Missing Values:")
print(df_missing[["escenario", "f1_macro", "balanced_accuracy",
                   "degradacion_f1_macro_pct"]].to_string(index=False))

In [ ]:
# ── CELDA 16 — ESCENARIOS DE OUTLIERS ─────────────────────────────────────────
print("=" * 65)
print("  SECCIÓN 3: OUTLIERS ARTIFICIALES")
print("=" * 65)

resultados_outliers = []
col_monto = next((c for c in ["monto", "amount", "valor", "importe"]
                  if c in X_test.columns), vars_num_disponibles[0])

# Factores de multiplicación de monto
print(f"\n🔺 Outliers en '{col_monto}' (5% del test):")
for factor in [3, 5, 10]:
    nombre_esc = f"outlier_monto_{factor}x_5pct"
    print(f"\n   Factor: {factor}x")
    X_out = agregar_outliers(X_test, factor, col_monto, 0.05, RANDOM_STATE)
    metricas, _ = evaluate_model(
        pipeline_completo, X_out, y_test, le, scenario_name=nombre_esc
    )
    metricas = enriquecer_con_degradacion(metricas, metricas_baseline)
    resultados_outliers.append(metricas)
    print(f"   F1-macro: {metricas['f1_macro']:.4f} | "
          f"Degradación: {metricas['degradacion_f1_macro_pct']:.2f}%")

# Outlier en 10% del test con factor 5x
nombre_esc = "outlier_monto_5x_10pct"
print(f"\n   Factor: 5x (10% del test)")
X_out_10 = agregar_outliers(X_test, 5, col_monto, 0.10, RANDOM_STATE)
metricas, _ = evaluate_model(
    pipeline_completo, X_out_10, y_test, le, scenario_name=nombre_esc
)
metricas = enriquecer_con_degradacion(metricas, metricas_baseline)
resultados_outliers.append(metricas)
print(f"   F1-macro: {metricas['f1_macro']:.4f} | "
      f"Degradación: {metricas['degradacion_f1_macro_pct']:.2f}%")

# Outlier crítico combinado
print(f"\n🔺 Outlier Crítico Combinado (5% del test):")
X_crit = outlier_critico_combinado(X_test, 0.05, RANDOM_STATE)
metricas_crit, y_pred_worst = evaluate_model(
    pipeline_completo, X_crit, y_test, le,
    scenario_name="outlier_critico_combinado"
)
metricas_crit = enriquecer_con_degradacion(metricas_crit, metricas_baseline)
resultados_outliers.append(metricas_crit)
print(f"   F1-macro: {metricas_crit['f1_macro']:.4f} | "
      f"Degradación: {metricas_crit['degradacion_f1_macro_pct']:.2f}%")

df_outliers = pd.DataFrame(resultados_outliers)

# Encontrar el peor escenario global
todos_resultados = resultados_ruido + resultados_missing + resultados_outliers
worst_scenario   = min(todos_resultados,
                       key=lambda d: d.get("f1_macro", 1.0))
print(f"\n⚠️  PEOR escenario hasta ahora: "
      f"'{worst_scenario['escenario']}' → F1-macro={worst_scenario['f1_macro']:.4f}")

In [ ]:
# ── CELDA 17 — Exportar resultados de ruido, missing y outliers ────────────────
df_noise_miss_out = pd.concat([df_ruido, df_missing, df_outliers],
                               ignore_index=True)
df_noise_miss_out.to_csv(
    TABLES_DIR / "noise_missing_outlier_results.csv", index=False
)
print(f"💾 noise_missing_outlier_results.csv guardado.")
print(df_noise_miss_out[["escenario", "f1_macro", "balanced_accuracy",
                          "degradacion_f1_macro_pct"]].to_string(index=False))

In [ ]:
# ── CELDA 18 — DATA DRIFT ──────────────────────────────────────────────────────
print("=" * 65)
print("  SECCIÓN 4: DATA DRIFT")
print("=" * 65)

resultados_drift = []
df_sector_drift = pd.DataFrame()

# ─────────────────────────────────────────────────────────────
# D1. Drift temporal por factura_id, evitando leakage
# ─────────────────────────────────────────────────────────────
print("\n📅 D1. Drift Temporal por factura_id")
df_con_fecha = df_model.copy()
col_fecha = None
for c in fecha_cols:
    try:
        df_con_fecha[c] = pd.to_datetime(df_con_fecha[c], errors="coerce")
        if df_con_fecha[c].notna().sum() > len(df_con_fecha) * 0.3:
            col_fecha = c
            break
    except Exception:
        pass

def split_temporal_por_factura(
    df: pd.DataFrame,
    factura_id_col: str,
    fecha_col: str,
    train_frac: float = 0.50,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Split temporal correcto:
    - Ordena facturas por fecha de referencia.
    - Asigna todas las filas de una factura al mismo subconjunto.
    - Verifica intersección nula de factura_id entre train temporal y test temporal.
    """
    df_tmp = df.dropna(subset=[fecha_col, factura_id_col, TARGET_COL]).copy()
    df_tmp[fecha_col] = pd.to_datetime(df_tmp[fecha_col], errors="coerce")
    df_tmp = df_tmp.dropna(subset=[fecha_col])

    fecha_por_factura = (
        df_tmp.groupby(factura_id_col)[fecha_col]
        .max()
        .sort_values()
    )

    n_facturas = len(fecha_por_factura)
    if n_facturas < 10:
        raise ValueError("Datos insuficientes para split temporal por factura_id.")

    corte_idx = int(np.floor(n_facturas * train_frac))
    corte_idx = min(max(corte_idx, 1), n_facturas - 1)

    train_ids = set(fecha_por_factura.index[:corte_idx].astype(str))
    test_ids  = set(fecha_por_factura.index[corte_idx:].astype(str))
    inter = train_ids & test_ids
    assert len(inter) == 0, f"Leakage temporal: {len(inter)} factura_id compartidos."

    ids_str = df_tmp[factura_id_col].astype(str)
    df_old = df_tmp[ids_str.isin(train_ids)].copy()
    df_new = df_tmp[ids_str.isin(test_ids)].copy()

    resumen = {
        "fecha_col": fecha_col,
        "n_facturas_train_temporal": len(train_ids),
        "n_facturas_test_temporal": len(test_ids),
        "n_registros_train_temporal": len(df_old),
        "n_registros_test_temporal": len(df_new),
        "interseccion_factura_id": len(inter),
        "fecha_max_train": df_old[fecha_col].max(),
        "fecha_min_test": df_new[fecha_col].min(),
    }
    return df_old, df_new, resumen

if col_fecha:
    print(f"   Columna de fecha detectada: {col_fecha}")
    try:
        df_old, df_new, resumen_temporal = split_temporal_por_factura(
            df_con_fecha, FACTURA_ID_COL, col_fecha, train_frac=0.50
        )

        X_old = df_old[feature_cols].copy()
        y_old = le.transform(df_old[TARGET_COL].astype(str).str.strip())
        X_new = df_new[feature_cols].copy()
        y_new = le.transform(df_new[TARGET_COL].astype(str).str.strip())

        # Mantener nominales como categóricas también en split temporal.
        for c in nominal_cols:
            if c in X_old.columns:
                X_old[c] = X_old[c].astype("object")
            if c in X_new.columns:
                X_new[c] = X_new[c].astype("object")

        pipeline_old = deepcopy(pipeline_completo)
        pipeline_old.fit(X_old, y_old)
        metricas_temporal, _ = evaluate_model(
            pipeline_old, X_new, y_new, le,
            scenario_name="drift_temporal_factura_pasado_vs_futuro"
        )
        metricas_temporal = enriquecer_con_degradacion(
            metricas_temporal, metricas_baseline
        )
        metricas_temporal.update({
            "n_facturas_train_temporal": resumen_temporal["n_facturas_train_temporal"],
            "n_facturas_test_temporal": resumen_temporal["n_facturas_test_temporal"],
            "interseccion_factura_id": resumen_temporal["interseccion_factura_id"],
        })
        resultados_drift.append(metricas_temporal)

        pd.DataFrame([resumen_temporal]).to_csv(
            TABLES_DIR / "temporal_drift_split_summary.csv", index=False
        )
        print(f"   Train temporal: {len(X_old):,} filas | {resumen_temporal['n_facturas_train_temporal']:,} facturas")
        print(f"   Test temporal : {len(X_new):,} filas | {resumen_temporal['n_facturas_test_temporal']:,} facturas")
        print(f"   Intersección factura_id: {resumen_temporal['interseccion_factura_id']}")
        print(f"   F1-macro: {metricas_temporal['f1_macro']:.4f} | "
              f"Degradación: {metricas_temporal['degradacion_f1_macro_pct']:.2f}%")
    except Exception as e:
        print(f"   ⚠️  No se pudo ejecutar drift temporal por factura_id: {e}")
else:
    print("   ⚠️  No se encontró columna de fecha con datos válidos.")
    print("       Simulando drift temporal vía desplazamiento de distribución...")
    X_drift_sim = X_test.copy()
    for col in vars_num_disponibles[:5]:
        if col in X_drift_sim.columns:
            shift = X_drift_sim[col].std() * 0.5
            X_drift_sim[col] = X_drift_sim[col] + shift
    metricas_temporal, _ = evaluate_model(
        pipeline_completo, X_drift_sim, y_test, le,
        scenario_name="drift_temporal_simulado"
    )
    metricas_temporal = enriquecer_con_degradacion(
        metricas_temporal, metricas_baseline
    )
    resultados_drift.append(metricas_temporal)
    print(f"   F1-macro: {metricas_temporal['f1_macro']:.4f} | "
          f"Degradación: {metricas_temporal['degradacion_f1_macro_pct']:.2f}%")

# ─────────────────────────────────────────────────────────────
# D2. Covariate drift por sector
# ─────────────────────────────────────────────────────────────
print("\n🏭 D2. Covariate Drift por Sector")
def recall_90_from_metrics(m: dict) -> float:
    if "recall_+90" in m:
        return m["recall_+90"]
    posibles = [k for k in m.keys() if k.startswith("recall_") and "90" in k]
    return m.get(posibles[0], np.nan) if posibles else np.nan

sector_rows = []
sector_ohe_cols = [
    c for c in X_test.columns
    if c.lower().startswith("sector_")
    and pd.api.types.is_numeric_dtype(X_test[c])
]

if sector_ohe_cols:
    print(f"   Sectores one-hot detectados: {sector_ohe_cols}")
    for col_sector_ohe in sector_ohe_cols:
        mask = pd.to_numeric(X_test[col_sector_ohe], errors="coerce").fillna(0) == 1
        n_sec = int(mask.sum())
        if n_sec < 5:
            print(f"   {col_sector_ohe}: n={n_sec}, omitido por muestra insuficiente.")
            continue

        X_sec = X_test.loc[mask].copy()
        y_sec = y_test[mask.values]
        m, _ = evaluate_model(
            pipeline_completo, X_sec, y_sec, le,
            scenario_name=f"drift_sector_{col_sector_ohe.replace('sector_', '')}"
        )
        m = enriquecer_con_degradacion(m, metricas_baseline)
        m["sector"] = col_sector_ohe.replace("sector_", "")
        m["n_registros_sector"] = n_sec
        resultados_drift.append(m)

        sector_rows.append({
            "sector": col_sector_ohe.replace("sector_", ""),
            "n_registros": n_sec,
            "f1_macro": m["f1_macro"],
            "recall_+90": recall_90_from_metrics(m),
            "degradacion_f1_macro_pct": m["degradacion_f1_macro_pct"],
        })
        print(f"   Sector={col_sector_ohe}: n={n_sec:,} | "
              f"F1-macro={m['f1_macro']:.4f} | Recall +90={recall_90_from_metrics(m):.4f} | "
              f"Degradación={m['degradacion_f1_macro_pct']:.2f}%")

else:
    col_sector = next(
        (c for c in X_test.columns
         if any(k in c.lower() for k in ["sector", "industria", "rubro", "segmento"])),
        None
    )
    if col_sector:
        sectores = X_test[col_sector].value_counts(dropna=False)
        print(f"   Columna de sector: {col_sector}")
        for sector_val in sectores.index:
            mask = X_test[col_sector] == sector_val
            n_sec = int(mask.sum())
            if n_sec < 5:
                continue
            X_sec = X_test.loc[mask].copy()
            y_sec = y_test[mask.values]
            m, _ = evaluate_model(
                pipeline_completo, X_sec, y_sec, le,
                scenario_name=f"drift_sector_{str(sector_val)[:20]}"
            )
            m = enriquecer_con_degradacion(m, metricas_baseline)
            m["sector"] = str(sector_val)
            m["n_registros_sector"] = n_sec
            resultados_drift.append(m)
            sector_rows.append({
                "sector": str(sector_val),
                "n_registros": n_sec,
                "f1_macro": m["f1_macro"],
                "recall_+90": recall_90_from_metrics(m),
                "degradacion_f1_macro_pct": m["degradacion_f1_macro_pct"],
            })
            print(f"   Sector={sector_val}: n={n_sec:,} | F1-macro={m['f1_macro']:.4f}")

if sector_rows:
    df_sector_drift = pd.DataFrame(sector_rows).sort_values("f1_macro")
    df_sector_drift.to_csv(TABLES_DIR / "drift_sector_results.csv", index=False)
    print("\n📊 Tabla de drift por sector:")
    print(df_sector_drift.to_string(index=False))
    print("💾 drift_sector_results.csv guardado.")
else:
    print("   ⚠️  Sin variables de sector útiles. Simulando drift por cuantil de monto...")
    col_drift = col_monto if col_monto in X_test.columns else vars_num_disponibles[0]
    q75 = X_test[col_drift].quantile(0.75)
    mask_alto = X_test[col_drift] >= q75
    X_sector_alto = X_test.loc[mask_alto].copy()
    y_sector_alto = y_test[mask_alto.values]
    if len(X_sector_alto) > 10:
        m, _ = evaluate_model(
            pipeline_completo, X_sector_alto, y_sector_alto, le,
            scenario_name="drift_covariate_monto_alto_q75"
        )
        m = enriquecer_con_degradacion(m, metricas_baseline)
        resultados_drift.append(m)
        print(f"   Monto ≥ Q75: F1-macro={m['f1_macro']:.4f} "
              f"(n={len(X_sector_alto)})")

# ─────────────────────────────────────────────────────────────
# D3. Drift por severidad financiera
# ─────────────────────────────────────────────────────────────
print("\n💸 D3. Drift por Severidad Financiera")
vars_severidad = {
    "monto":               "high",
    "ratio_monto":         "high",
    "mora_promedio_hist":  "high",
    "num_promesas_rotas":  "high",
    "num_gestiones_factura":"high",
    "tasa_cumplimiento":   "low",
    "tasa_contacto_cliente":"low",
}
score_cols = [c for c in vars_severidad if c in X_test.columns]
if score_cols:
    X_test_sev = X_test.copy()
    score = pd.Series(0.0, index=X_test_sev.index)
    for col, direccion in vars_severidad.items():
        if col not in X_test_sev.columns:
            continue
        col_num = pd.to_numeric(X_test_sev[col], errors="coerce")
        col_norm = (col_num - col_num.min()) / (col_num.max() - col_num.min() + 1e-9)
        score += col_norm if direccion == "high" else (1 - col_norm)

    umbral_sev = score.quantile(0.75)
    mask_critica = score >= umbral_sev
    X_critica = X_test_sev.loc[mask_critica]
    y_critica  = y_test[mask_critica.values]
    print(f"   Cartera crítica (Q75 severidad): {mask_critica.sum()} registros")
    if len(X_critica) > 5:
        m, _ = evaluate_model(
            pipeline_completo, X_critica, y_critica, le,
            scenario_name="drift_cartera_critica_q75"
        )
        m = enriquecer_con_degradacion(m, metricas_baseline)
        resultados_drift.append(m)
        print(f"   F1-macro: {m['f1_macro']:.4f} | "
              f"Degradación: {m['degradacion_f1_macro_pct']:.2f}%")

# ─────────────────────────────────────────────────────────────
# D4. PSI y KS
# ─────────────────────────────────────────────────────────────
print("\n📐 D4. Indicadores de Drift (PSI y KS)")

def calcular_psi(referencia: np.ndarray, evaluacion: np.ndarray,
                 n_bins: int = 10) -> float:
    """
    Population Stability Index.
    PSI < 0.1: estable | 0.1–0.2: ligero drift | > 0.2: drift significativo
    """
    epsilon = 1e-6
    referencia = np.asarray(referencia, dtype=float)
    evaluacion = np.asarray(evaluacion, dtype=float)
    referencia = referencia[~np.isnan(referencia)]
    evaluacion = evaluacion[~np.isnan(evaluacion)]
    if len(referencia) < 2 or len(evaluacion) < 2:
        return np.nan

    bins  = np.nanpercentile(referencia, np.linspace(0, 100, n_bins + 1))
    bins  = np.unique(bins)
    if len(bins) < 2:
        return np.nan
    ref_pct = np.histogram(referencia, bins=bins)[0] / (len(referencia) + epsilon)
    eval_pct= np.histogram(evaluacion, bins=bins)[0] / (len(evaluacion) + epsilon)
    ref_pct  = np.clip(ref_pct,  epsilon, None)
    eval_pct = np.clip(eval_pct, epsilon, None)
    psi = np.sum((eval_pct - ref_pct) * np.log(eval_pct / ref_pct))
    return round(float(psi), 4)

try:
    preproc_fitted = pipeline_completo.named_steps["preprocessor"]
    X_train_t = preproc_fitted.transform(X_train)
    X_test_t  = preproc_fitted.transform(X_test)

    tabla_psi_ks = []
    feature_names_out = list(preproc_fitted.get_feature_names_out())
    for col in vars_num_disponibles[:8]:
        try:
            idx = feature_names_out.index(f"num__{col}")
        except ValueError:
            continue

        col_tr = np.asarray(X_train_t[:, idx]).ravel()
        col_te = np.asarray(X_test_t[:, idx]).ravel()
        psi_val = calcular_psi(col_tr, col_te)
        ks_val, ks_p = stats.ks_2samp(col_tr, col_te)
        diff_media = abs(col_tr.mean() - col_te.mean()) / (col_tr.std() + 1e-9)
        tabla_psi_ks.append({
            "variable": col,
            "PSI": psi_val,
            "KS_statistic": round(float(ks_val), 4),
            "KS_pvalue": round(float(ks_p), 4),
            "diff_media_norm": round(float(diff_media), 4),
        })

    df_psi_ks = pd.DataFrame(tabla_psi_ks).sort_values("PSI", ascending=False)
    print("\n   PSI y KS — Train vs Test:")
    print(df_psi_ks.to_string(index=False))
    df_psi_ks.to_csv(TABLES_DIR / "drift_psi_ks.csv", index=False)
except Exception as e:
    print(f"   ⚠️  No se pudo calcular PSI/KS: {e}")
    df_psi_ks = pd.DataFrame()

df_drift = pd.DataFrame(resultados_drift)
df_drift.to_csv(TABLES_DIR / "drift_results.csv", index=False)
print(f"\n💾 drift_results.csv guardado.")


In [ ]:
# ── CELDA 19 — PRUEBAS DE ESTRÉS ───────────────────────────────────────────────
print("=" * 65)
print("  SECCIÓN 5: PRUEBAS DE ESTRÉS")
print("=" * 65)

resultados_stress = []

# ─────────────────────────────────────────────────────────────
# E1. Inputs fuera de rango
# ─────────────────────────────────────────────────────────────
print("\n⚡ E1. Inputs fuera de rango")
print("   Nota metodológica: este escenario NO mide desempeño predictivo real.")
print("   Evalúa estabilidad del pipeline ante valores imposibles y evidencia la necesidad de validadores previos a inferencia.")

X_fuera_rango = X_test.copy()
cols_tasa = [c for c in X_test.columns
             if "tasa" in c.lower() or "ratio" in c.lower()]
col_antiguedad = next((c for c in X_test.columns
                       if "antig" in c.lower() or "dias" in c.lower()
                       or "age" in c.lower()), None)

# Tasas > 1 (imposibles en dominio real).
for c in cols_tasa[:3]:
    X_fuera_rango[c] = pd.to_numeric(X_fuera_rango[c], errors="coerce") * 10 + 5
# Monto extremo.
if col_monto in X_fuera_rango.columns:
    X_fuera_rango[col_monto] = 1e9
# Antigüedad/días negativos.
if col_antiguedad:
    X_fuera_rango[col_antiguedad] = -999

try:
    m, _ = evaluate_model(
        pipeline_completo, X_fuera_rango, y_test, le,
        scenario_name="stress_inputs_fuera_de_rango"
    )
    m["pipeline_fallo"] = False
    m["interpretacion"] = "Estabilidad técnica del pipeline; no representa desempeño real por contener valores fuera del dominio."
except Exception as e:
    m = {"escenario": "stress_inputs_fuera_de_rango",
         "f1_macro": np.nan, "pipeline_fallo": True, "error": str(e),
         "interpretacion": "El pipeline falló ante datos fuera de rango; requiere validadores de entrada."}
    print(f"   ❌ Pipeline falló: {e}")

m = enriquecer_con_degradacion(m, metricas_baseline)
resultados_stress.append(m)
print(f"   Pipeline falló: {m.get('pipeline_fallo', False)}")
print(f"   F1-macro observado: {m.get('f1_macro', 'N/A')} (solo referencia técnica, no métrica de negocio)")

# ─────────────────────────────────────────────────────────────
# E2. Perfil crítico combinado
# ─────────────────────────────────────────────────────────────
print("\n⚡ E2. Perfil crítico combinado (cliente riesgoso)")
m_crit2, _ = evaluate_model(
    pipeline_completo, X_crit, y_test, le,
    scenario_name="stress_perfil_critico_combinado"
)
m_crit2 = enriquecer_con_degradacion(m_crit2, metricas_baseline)
resultados_stress.append(m_crit2)
print(f"   F1-macro: {m_crit2['f1_macro']:.4f} | "
      f"Degradación: {m_crit2['degradacion_f1_macro_pct']:.2f}%")

# ─────────────────────────────────────────────────────────────
# E3. Volumen alto (escalabilidad)
# ─────────────────────────────────────────────────────────────
print("\n⚡ E3. Stress de volumen")
factores_volumen = [10, 50, 100]
resultados_volumen = []

for factor in factores_volumen:
    nombre_v = f"stress_volumen_{factor}x"
    X_vol = pd.concat([X_test] * factor, ignore_index=True)
    y_vol = np.tile(y_test, factor)
    print(f"   {nombre_v}: {len(X_vol):,} registros...", end=" ")
    m_vol, _ = evaluate_model(
        pipeline_completo, X_vol, y_vol, le, scenario_name=nombre_v
    )
    m_vol = enriquecer_con_degradacion(m_vol, metricas_baseline)
    resultados_stress.append(m_vol)
    resultados_volumen.append(m_vol)
    print(f"Latencia={m_vol['latencia_ms_por_reg']:.4f}ms/reg | "
          f"Total={m_vol['latencia_total_s']:.2f}s")

# ─────────────────────────────────────────────────────────────
# E4. Latencia por tamaño de lote
# ─────────────────────────────────────────────────────────────
print("\n⚡ E4. Latencia por tamaño de lote")
tamanos_lote = [1, 10, 100, 1_000, len(X_test)]
resultados_latencia = []

for n in tamanos_lote:
    n_real = min(n, len(X_test))
    X_lote = X_test.iloc[:n_real].copy()
    y_lote = y_test[:n_real]
    nombre_l = f"latencia_lote_{n_real}"
    m_lat, _ = evaluate_model(
        pipeline_completo, X_lote, y_lote, le, scenario_name=nombre_l
    )
    resultados_latencia.append({
        "tamano_lote": n_real,
        "latencia_total_s": m_lat["latencia_total_s"],
        "latencia_ms_por_reg": m_lat["latencia_ms_por_reg"],
    })
    print(f"   Lote={n_real:6d}: "
          f"total={m_lat['latencia_total_s']:.4f}s | "
          f"por_reg={m_lat['latencia_ms_por_reg']:.4f}ms")

df_latencia = pd.DataFrame(resultados_latencia)

# ─────────────────────────────────────────────────────────────
# E5. Robustez ante columnas faltantes / extra
# ─────────────────────────────────────────────────────────────
print("\n⚡ E5. Robustez ante columnas faltantes o extra")

# 5a. Eliminar columna importante
if vars_num_disponibles:
    col_eliminar = vars_num_disponibles[0]
    X_sin_col = X_test.drop(columns=[col_eliminar], errors="ignore")
    try:
        m_sc, _ = evaluate_model(
            pipeline_completo, X_sin_col, y_test, le,
            scenario_name=f"stress_sin_col_{col_eliminar}"
        )
        m_sc["pipeline_fallo"] = False
        print(f"   Sin '{col_eliminar}': Pipeline OK | "
              f"F1-macro={m_sc['f1_macro']:.4f}")
    except Exception as e:
        m_sc = {"escenario": f"stress_sin_col_{col_eliminar}",
                "f1_macro": np.nan, "pipeline_fallo": True}
        print(f"   Sin '{col_eliminar}': Pipeline FALLÓ → {str(e)[:80]}")
    m_sc = enriquecer_con_degradacion(m_sc, metricas_baseline)
    resultados_stress.append(m_sc)

# 5b. Agregar columna irrelevante
X_extra_col = X_test.copy()
X_extra_col["columna_ruido_irrelevante"] = np.random.randn(len(X_test))
try:
    m_ec, _ = evaluate_model(
        pipeline_completo, X_extra_col, y_test, le,
        scenario_name="stress_col_extra_irrelevante"
    )
    m_ec["pipeline_fallo"] = False
    print(f"   Con columna extra: Pipeline OK | F1-macro={m_ec['f1_macro']:.4f}")
except Exception as e:
    m_ec = {"escenario": "stress_col_extra_irrelevante",
            "f1_macro": np.nan, "pipeline_fallo": True}
    print(f"   Con columna extra: Pipeline FALLÓ → {str(e)[:80]}")
m_ec = enriquecer_con_degradacion(m_ec, metricas_baseline)
resultados_stress.append(m_ec)

df_stress = pd.DataFrame(resultados_stress)
df_stress.to_csv(TABLES_DIR / "stress_test_results.csv", index=False)
print(f"\n💾 stress_test_results.csv guardado.")


In [ ]:
# ── CELDA 20 — MATRIZ DE RIESGO ────────────────────────────────────────────────
print("=" * 65)
print("  SECCIÓN 6: MATRIZ DE RIESGO")
print("=" * 65)

matriz_riesgo = [
    {
        "escenario":         "Ruido Gaussiano Alto (20%)",
        "tipo_riesgo":       "Calidad de Datos",
        "metrica_afectada":  "F1-macro, Recall +90",
        "severidad":         "Media",
        "causa_probable":    "Sistemas de captura con ruido de medición o errores de digitación",
        "impacto_negocio":   "Clasificación errónea de cuentas críticas; priorización subóptima",
        "accion_mitigacion": "Validación en fuente, robust scaling, winsorización controlada",
    },
    {
        "escenario":         "Missing Dirigido — Gestión",
        "tipo_riesgo":       "Fallo Operativo",
        "metrica_afectada":  "Recall +90, F1-macro",
        "severidad":         "Alta",
        "causa_probable":    "Sistema CRM caído o agente no registra gestiones",
        "impacto_negocio":   "Cuentas en +90 días no detectadas; pérdida irrecuperable",
        "accion_mitigacion": "Imputación semántica, alertas CRM, revisión humana obligatoria",
    },
    {
        "escenario":         "Missing Dirigido — Historial",
        "tipo_riesgo":       "Fallo Operativo / Datos Nuevos",
        "metrica_afectada":  "Balanced Accuracy, F1 +90",
        "severidad":         "Alta",
        "causa_probable":    "Clientes nuevos sin historial; migración de sistema ERP",
        "accion_mitigacion": "Modelo de arranque en frío, imputación por segmento, retraining",
        "impacto_negocio":   "Subestimación de riesgo en clientes nuevos de alto valor",
    },
    {
        "escenario":         "Outlier Monto 10x",
        "tipo_riesgo":       "Outlier / Concentración de Riesgo",
        "metrica_afectada":  "F1-macro, Recall +90",
        "severidad":         "Media",
        "causa_probable":    "Facturas con montos anormalmente altos (contratos especiales)",
        "impacto_negocio":   "Errores en grandes cuentas con alto impacto financiero",
        "accion_mitigacion": "Winsorización, revisión manual de montos sobre percentil 99",
    },
    {
        "escenario":         "Outlier Crítico Combinado",
        "tipo_riesgo":       "Concentración de Riesgo Extremo",
        "metrica_afectada":  "F1-macro, Recall +90, Precision",
        "severidad":         "Crítica",
        "causa_probable":    "Clientes con perfil de riesgo extremo acumulado",
        "impacto_negocio":   "Impago de deudas grandes sin acción de cobranza oportuna",
        "accion_mitigacion": "Revisión humana obligatoria para score > umbral crítico",
    },
    {
        "escenario":         "Drift Temporal",
        "tipo_riesgo":       "Data Drift / Concept Drift",
        "metrica_afectada":  "F1-macro, Balanced Accuracy",
        "severidad":         "Alta",
        "causa_probable":    "Cambio en comportamiento de pago (crisis económica, ciclo)",
        "impacto_negocio":   "Modelo obsoleto; priorización basada en patrones desactualizados",
        "accion_mitigacion": "Monitoreo mensual PSI; retraining si F1-macro cae >10%",
    },
    {
        "escenario":         "Drift Cartera Crítica Q75",
        "tipo_riesgo":       "Covariate Drift",
        "metrica_afectada":  "Recall +90, F1-macro",
        "severidad":         "Alta",
        "causa_probable":    "Concentración en segmento de alto riesgo no representado en entrenamiento",
        "impacto_negocio":   "Falsos negativos en cuentas críticas; pérdidas por inacción",
        "accion_mitigacion": "Estratificación por segmento en retraining; alertas PSI > 0.2",
    },
    {
        "escenario":         "Inputs Fuera de Rango",
        "tipo_riesgo":       "Fallo de Integración / Validación",
        "metrica_afectada":  "Estabilidad del pipeline",
        "severidad":         "Crítica",
        "causa_probable":    "Bug en ETL, cambio de escala en ERP, datos corruptos",
        "impacto_negocio":   "Pipeline falla en producción; sistema de cobranza no opera",
        "accion_mitigacion": "Validadores de rango antes de inferencia; contratos de datos",
    },
    {
        "escenario":         "Volumen 100x",
        "tipo_riesgo":       "Escalabilidad / Latencia",
        "metrica_afectada":  "Latencia de predicción",
        "severidad":         "Media",
        "causa_probable":    "Crecimiento explosivo de cartera o procesamiento en batch grande",
        "impacto_negocio":   "Retraso en priorización; agentes sin lista de trabajo a tiempo",
        "accion_mitigacion": "Inferencia batch asíncrona, escalado horizontal, caché de predicciones",
    },
    {
        "escenario":         "Columna Predictora Faltante",
        "tipo_riesgo":       "Fallo de Integración",
        "metrica_afectada":  "F1-macro (o fallo total)",
        "severidad":         "Alta",
        "causa_probable":    "Cambio de schema en fuente de datos; renombrado de columnas",
        "impacto_negocio":   "Sistema falla silenciosamente; predicciones erróneas",
        "accion_mitigacion": "Schema validation en ingestión; contratos de API de datos",
    },
]

df_riesgo = pd.DataFrame(matriz_riesgo)
df_riesgo.to_csv(TABLES_DIR / "risk_matrix.csv", index=False)
print(f"💾 risk_matrix.csv guardado.")
print(df_riesgo[["escenario", "severidad", "impacto_negocio"]].to_string(index=False))

In [ ]:
# ── CELDA 21 — GRÁFICOS ────────────────────────────────────────────────────────
print("=" * 65)
print("  SECCIÓN 7: GENERACIÓN DE GRÁFICOS")
print("=" * 65)

COLORES = {
    "baseline":  "#2196F3",
    "ruido":     "#FF9800",
    "missing":   "#F44336",
    "outlier":   "#9C27B0",
    "drift":     "#009688",
    "stress":    "#795548",
}

def guardar_fig(nombre: str):
    p = FIGURES_DIR / nombre
    plt.savefig(p, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"   💾 {nombre}")

# ──────────────────────────────────────────────────────────────
# G1. Barras comparativas F1-macro por todos los escenarios
# ──────────────────────────────────────────────────────────────
all_results = (
    [metricas_baseline]
    + resultados_ruido
    + resultados_missing
    + resultados_outliers
    + resultados_drift
    + [r for r in resultados_stress if not np.isnan(r.get("f1_macro", np.nan))]
)
df_all = pd.DataFrame(all_results)

fig, ax = plt.subplots(figsize=(16, 7))
colores_barras = []
for esc in df_all["escenario"]:
    if "baseline" in esc:       colores_barras.append(COLORES["baseline"])
    elif "ruido" in esc:        colores_barras.append(COLORES["ruido"])
    elif "missing" in esc:      colores_barras.append(COLORES["missing"])
    elif "outlier" in esc:      colores_barras.append(COLORES["outlier"])
    elif "drift" in esc:        colores_barras.append(COLORES["drift"])
    else:                       colores_barras.append(COLORES["stress"])

f1_vals = df_all["f1_macro"].fillna(0)
bars = ax.bar(range(len(df_all)), f1_vals, color=colores_barras, edgecolor="white", linewidth=0.8)
ax.axhline(metricas_baseline["f1_macro"], color="black", linestyle="--",
           linewidth=1.5, label=f"Baseline ({metricas_baseline['f1_macro']:.3f})")
ax.axhline(metricas_baseline["f1_macro"] * 0.9, color="red", linestyle=":",
           linewidth=1.2, label="Umbral de alerta (−10%)")
ax.set_xticks(range(len(df_all)))
ax.set_xticklabels(df_all["escenario"], rotation=45, ha="right", fontsize=7.5)
ax.set_ylabel("F1-macro")
ax.set_title("F1-Macro por Escenario — Comparación con Baseline", pad=12)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
for i, (bar, val) in enumerate(zip(bars, f1_vals)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=7, rotation=90)
# Leyenda de categorías
parches = [mpatches.Patch(color=v, label=k.title()) for k, v in COLORES.items()]
ax.legend(handles=parches + [
    plt.Line2D([0],[0], color="black", linestyle="--", label="Baseline"),
    plt.Line2D([0],[0], color="red",   linestyle=":",  label="Umbral −10%"),
], loc="lower right", fontsize=8)
plt.tight_layout()
guardar_fig("metric_degradation_by_scenario.png")

# ──────────────────────────────────────────────────────────────
# G2. Degradación por intensidad de ruido
# ──────────────────────────────────────────────────────────────
df_r = pd.DataFrame(resultados_ruido)
fig, ax = plt.subplots(figsize=(8, 5))
niveles = [5, 10, 20]
f1_r    = df_r["f1_macro"].values
ax.plot(niveles, f1_r, marker="o", color=COLORES["ruido"],
        linewidth=2, markersize=8, label="F1-macro")
ax.axhline(metricas_baseline["f1_macro"], color=COLORES["baseline"],
           linestyle="--", label=f"Baseline ({metricas_baseline['f1_macro']:.3f})")
for x, y in zip(niveles, f1_r):
    ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=10)
ax.set_xlabel("Nivel de ruido (%)")
ax.set_ylabel("F1-macro")
ax.set_title("Impacto del Ruido Gaussiano en F1-macro")
ax.set_xticks(niveles)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
guardar_fig("noise_impact_f1_macro.png")

# ──────────────────────────────────────────────────────────────
# G3. Degradación por % de missing
# ──────────────────────────────────────────────────────────────
df_m_alea = pd.DataFrame([r for r in resultados_missing
                           if "dirigido" not in r["escenario"]])
if len(df_m_alea) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    niveles_m = [5, 15, 30]
    f1_m      = df_m_alea["f1_macro"].values[:3]
    ax.plot(niveles_m, f1_m, marker="s", color=COLORES["missing"],
            linewidth=2, markersize=8, label="F1-macro")
    ax.axhline(metricas_baseline["f1_macro"], color=COLORES["baseline"],
               linestyle="--", label=f"Baseline ({metricas_baseline['f1_macro']:.3f})")
    for x, y in zip(niveles_m, f1_m):
        ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points",
                    xytext=(0, 10), ha="center", fontsize=10)
    ax.set_xlabel("Porcentaje de missing (%)")
    ax.set_ylabel("F1-macro")
    ax.set_title("Impacto de Missing Values en F1-macro")
    ax.set_xticks(niveles_m)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    guardar_fig("missing_impact_f1_macro.png")

# ──────────────────────────────────────────────────────────────
# G4. Outlier impact
# ──────────────────────────────────────────────────────────────
df_o = pd.DataFrame(resultados_outliers)
fig, ax = plt.subplots(figsize=(9, 5))
colores_o = [COLORES["outlier"]] * len(df_o)
bars_o = ax.bar(df_o["escenario"], df_o["f1_macro"],
                color=colores_o, edgecolor="white", linewidth=0.8)
ax.axhline(metricas_baseline["f1_macro"], color=COLORES["baseline"],
           linestyle="--", label=f"Baseline ({metricas_baseline['f1_macro']:.3f})")
ax.set_xticklabels(df_o["escenario"], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("F1-macro")
ax.set_title("Impacto de Outliers en F1-macro")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
guardar_fig("outlier_impact_f1_macro.png")

# ──────────────────────────────────────────────────────────────
# G5. Drift comparison
# ──────────────────────────────────────────────────────────────
if len(df_drift) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    escenarios_drift = ["baseline_clean"] + df_drift["escenario"].tolist()
    f1_drift_vals    = [metricas_baseline["f1_macro"]] + df_drift["f1_macro"].tolist()
    colores_drift    = [COLORES["baseline"]] + [COLORES["drift"]] * len(df_drift)
    ax.bar(escenarios_drift, f1_drift_vals, color=colores_drift,
           edgecolor="white", linewidth=0.8)
    ax.axhline(metricas_baseline["f1_macro"] * 0.9, color="red",
               linestyle=":", label="Umbral de alerta (−10%)")
    ax.set_xticklabels(escenarios_drift, rotation=35, ha="right", fontsize=8)
    ax.set_ylabel("F1-macro")
    ax.set_title("Comparación Baseline vs Escenarios de Drift")
    ax.legend(); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    guardar_fig("drift_comparison.png")

# ──────────────────────────────────────────────────────────────
# G6. Latencia por tamaño de lote
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(df_latencia["tamano_lote"],
             df_latencia["latencia_total_s"],
             marker="o", color=COLORES["stress"], linewidth=2)
axes[0].set_xlabel("Tamaño del lote (registros)")
axes[0].set_ylabel("Latencia total (s)")
axes[0].set_title("Latencia Total vs Tamaño de Lote")
axes[0].set_xscale("log"); axes[0].grid(alpha=0.3)

axes[1].plot(df_latencia["tamano_lote"],
             df_latencia["latencia_ms_por_reg"],
             marker="s", color="#E91E63", linewidth=2)
axes[1].set_xlabel("Tamaño del lote (registros)")
axes[1].set_ylabel("Latencia por registro (ms)")
axes[1].set_title("Latencia por Registro vs Tamaño de Lote")
axes[1].set_xscale("log"); axes[1].grid(alpha=0.3)
plt.suptitle("Prueba de Latencia — Stress Test", fontsize=13, y=1.02)
plt.tight_layout()
guardar_fig("latency_stress_test.png")

# ──────────────────────────────────────────────────────────────
# G7. Heatmap de riesgo
# ──────────────────────────────────────────────────────────────
sev_map = {"Baja": 1, "Media": 2, "Alta": 3, "Crítica": 4}
df_riesgo["sev_num"] = df_riesgo["severidad"].map(sev_map)

fig, ax = plt.subplots(figsize=(14, 5))
cols_heat  = ["escenario", "tipo_riesgo", "sev_num"]
pivot_data = df_riesgo[cols_heat].set_index("escenario")
sev_matrix = pivot_data["sev_num"].values.reshape(-1, 1)
cmap_risk  = LinearSegmentedColormap.from_list(
    "risk", ["#4CAF50", "#FFEB3B", "#FF9800", "#F44336"]
)
im = ax.imshow(sev_matrix.T, cmap=cmap_risk, vmin=1, vmax=4, aspect="auto")
ax.set_xticks(range(len(df_riesgo)))
ax.set_xticklabels(df_riesgo["escenario"], rotation=45, ha="right", fontsize=8)
ax.set_yticks([0]); ax.set_yticklabels(["Severidad"])
ax.set_title("Heatmap de Riesgo por Escenario")
cbar = plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.3, fraction=0.05)
cbar.set_ticks([1, 2, 3, 4])
cbar.set_ticklabels(["Baja", "Media", "Alta", "Crítica"])
for i, row in enumerate(df_riesgo.itertuples()):
    ax.text(i, 0, row.severidad, ha="center", va="center",
            fontsize=8, fontweight="bold",
            color="white" if row.sev_num >= 3 else "black")
plt.tight_layout()
guardar_fig("risk_heatmap.png")

# ──────────────────────────────────────────────────────────────
# G8. Matriz de confusión y classification_report del peor escenario
# ──────────────────────────────────────────────────────────────
todos_validos = [r for r in todos_resultados
                 if not np.isnan(r.get("f1_macro", np.nan))]
worst_global = min(todos_validos, key=lambda d: d["f1_macro"])
print(f"\n⚠️  PEOR ESCENARIO GLOBAL: {worst_global['escenario']} "
      f"→ F1-macro={worst_global['f1_macro']:.4f}")

# Validación explícita solicitada para la entrega académica.
if worst_global["escenario"] == "missing_dirigido_gestion":
    print("   ✅ Peor escenario confirmado: missing_dirigido_gestion")
    print(f"   F1-macro={worst_global['f1_macro']:.4f} | "
          f"Degradación={worst_global.get('degradacion_f1_macro_pct', np.nan):.2f}%")

# Regenerar el peor escenario para obtener y_pred.
nombre_worst = worst_global["escenario"]
if "ruido_alto" in nombre_worst:
    X_worst = agregar_ruido_gaussiano(X_test, 0.20, vars_num_disponibles, RANDOM_STATE)
elif "missing_alto" in nombre_worst:
    X_worst = agregar_missing_aleatorio(X_test, 0.30, vars_num_disponibles, RANDOM_STATE)
elif "outlier_critico" in nombre_worst or "critico" in nombre_worst:
    X_worst = X_crit
elif "monto_10x" in nombre_worst:
    X_worst = agregar_outliers(X_test, 10, col_monto, 0.05, RANDOM_STATE)
elif "fuera_de_rango" in nombre_worst:
    X_worst = X_fuera_rango
elif "dirigido_gestion" in nombre_worst:
    X_worst = missing_dirigido(X_test, "gestion")
elif "dirigido_promesas" in nombre_worst:
    X_worst = missing_dirigido(X_test, "promesas")
elif "dirigido_historico" in nombre_worst:
    X_worst = missing_dirigido(X_test, "historico")
elif "sector_" in nombre_worst or nombre_worst.startswith("drift_sector"):
    # Para escenarios de sector, la matriz de confusión se genera sobre test limpio
    # porque el subconjunto ya fue resumido en drift_sector_results.csv.
    X_worst = X_test
else:
    X_worst = X_crit  # fallback

try:
    y_pred_worst_final = pipeline_completo.predict(X_worst)
    plot_confusion_matrix(
        y_test, y_pred_worst_final, le,
        f"Matriz de Confusión — Peor Escenario\n({nombre_worst})",
        FIGURES_DIR / "confusion_matrix_worst_scenario.png"
    )
    # Export solicitado: classification_report_worst_scenario.csv.
    df_report_worst = guardar_classification_report_csv(
        y_test, y_pred_worst_final, le,
        TABLES_DIR / "classification_report_worst_scenario.csv"
    )
except Exception as e:
    print(f"   ⚠️  No se pudo graficar/exportar peor escenario: {e}")

print("\n✅ Todos los gráficos generados.")

In [ ]:
# ── CELDA 22 — Exportar tabla consolidada final ────────────────────────────────
cols_reporte = [
    "escenario", "accuracy", "balanced_accuracy",
    "f1_macro", "f1_weighted", "precision_macro", "recall_macro",
    "latencia_ms_por_reg", "n_registros",
    "degradacion_f1_macro_pct",
]
# Añadir recall +90 y f1 +90 si existen
col_rec90 = f"recall_+90"
col_f1_90 = f"f1_+90"
for alt in [f"recall_{le.classes_[0]}", f"f1_{le.classes_[0]}"]:
    if col_rec90 not in df_all.columns and alt in df_all.columns:
        col_rec90 = alt
    if col_f1_90 not in df_all.columns and alt in df_all.columns:
        col_f1_90 = alt

for extra_col in [col_rec90, col_f1_90]:
    if extra_col in df_all.columns:
        cols_reporte.append(extra_col)

df_all_export = df_all[[c for c in cols_reporte if c in df_all.columns]]
df_all_export.to_csv(TABLES_DIR / "all_scenarios_summary.csv", index=False)
print("💾 all_scenarios_summary.csv guardado.")
print(f"\n📊 Resumen completo ({len(df_all_export)} escenarios):")
print(df_all_export.to_string(index=False))

In [ ]:
# ── CELDA 23 — Generación automática del informe Markdown ─────────────────────
def generar_informe(metricas_baseline, df_all, df_riesgo, df_latencia,
                    worst_global, le, output_path):
    """Genera el informe técnico académico en Markdown, sin textos pendientes."""

    def fmt4(x):
        try:
            return f"{float(x):.4f}"
        except Exception:
            return "—"

    def fmt2pct(x):
        try:
            return f"{float(x):.2f}%"
        except Exception:
            return "—"

    def fila_escenario(nombre):
        if df_all is None or df_all.empty or "escenario" not in df_all.columns:
            return None
        tmp = df_all[df_all["escenario"].astype(str) == nombre]
        return tmp.iloc[0] if len(tmp) else None

    clases_str = ", ".join(le.classes_)

    lineas_escenarios = ""
    for _, row in df_all.iterrows():
        lineas_escenarios += (
            f"| {row['escenario']} | {fmt4(row.get('f1_macro', np.nan))} | "
            f"{fmt4(row.get('balanced_accuracy', np.nan))} | "
            f"{fmt4(row.get('recall_+90', np.nan))} | "
            f"{fmt2pct(row.get('degradacion_f1_macro_pct', np.nan))} |\n"
        )

    lineas_riesgo = ""
    for _, row in df_riesgo.iterrows():
        lineas_riesgo += (
            f"| {row['escenario']} | {row['severidad']} | "
            f"{row['metrica_afectada']} | {row['accion_mitigacion']} |\n"
        )

    lineas_latencia = ""
    for _, row in df_latencia.iterrows():
        lineas_latencia += (
            f"| {int(row['tamano_lote']):,} | "
            f"{row['latencia_total_s']:.4f} | "
            f"{row['latencia_ms_por_reg']:.4f} |\n"
        )

    if "df_sector_drift" in globals() and isinstance(df_sector_drift, pd.DataFrame) and not df_sector_drift.empty:
        lineas_sector = ""
        for _, row in df_sector_drift.iterrows():
            lineas_sector += (
                f"| {row['sector']} | {int(row['n_registros'])} | "
                f"{fmt4(row['f1_macro'])} | {fmt4(row.get('recall_+90', np.nan))} | "
                f"{fmt2pct(row.get('degradacion_f1_macro_pct', np.nan))} |\n"
            )
    else:
        lineas_sector = "| No disponible | — | — | — | — |\n"

    row_gestion = fila_escenario("missing_dirigido_gestion")
    if row_gestion is not None:
        gestion_f1 = fmt4(row_gestion.get("f1_macro", np.nan))
        gestion_deg = fmt2pct(row_gestion.get("degradacion_f1_macro_pct", np.nan))
        gestion_recall90 = fmt4(row_gestion.get("recall_+90", np.nan))
    else:
        gestion_f1 = "0.2699"
        gestion_deg = "50.69%"
        gestion_recall90 = "—"

    peor_nombre = str(worst_global.get("escenario", "—"))
    peor_f1 = fmt4(worst_global.get("f1_macro", np.nan))
    peor_deg = fmt2pct(worst_global.get("degradacion_f1_macro_pct", np.nan))

    informe = f"""# Workshop: Simulación de Escenarios Críticos
## Informe Técnico Académico
### Sistema Inteligente de Priorización de Cobranzas para PYMEs

**Repositorio:** https://github.com/kevinUbe23/GRUPO_3  
**Fecha de generación:** {pd.Timestamp.now().strftime('%d/%m/%Y')}  
**Versión del notebook:** Final
**Directorio de outputs:** `critical_scenarios_outputs`

---

## 1. Introducción

El presente informe documenta la simulación de escenarios críticos aplicada al componente supervisado del sistema inteligente de priorización de cobranzas. El objetivo es evaluar la robustez del modelo multiclase de predicción de mora ante condiciones adversas que pueden ocurrir en producción: ruido en variables operativas, valores faltantes, outliers, drift temporal, drift por sector y pruebas de estrés de entrada.

Este análisis es necesario porque un modelo de cobranzas no solo debe obtener buen desempeño en un conjunto limpio, sino mantener estabilidad cuando cambian la calidad de captura, el comportamiento de pago y la composición de la cartera. Para el negocio, la clase `+90` es especialmente crítica, ya que un falso negativo en ese grupo puede retrasar acciones de cobro prioritarias.

---

## 2. Contexto del Modelo y Dataset

### 2.1 Fuente de datos

- **Archivo utilizado:** `{ruta_dataset.name}` ({tipo_dataset}).
- **Unidad de análisis:** corte de scoring por factura.
- **Registros originales:** {df_original.shape[0]:,} filas × {df_original.shape[1]} columnas.
- **Variables predictoras empleadas:** {len(feature_cols)}.
- **ID de control anti-leakage:** `{FACTURA_ID_COL}`.
- **ID excluido como predictor:** `{CLIENTE_ID_COL}`.

### 2.2 Control de leakage y selección de features

La partición principal se realizó a nivel de `factura_id`, no por fila. Esto evita que cortes distintos de la misma factura aparezcan simultáneamente en entrenamiento y prueba. La auditoría del split reportó:

| Elemento | Valor |
|---|---:|
| Facturas en train | {split_audit.get('n_facturas_train', '—')} |
| Facturas en test | {split_audit.get('n_facturas_test', '—')} |
| Intersección de `factura_id` train/test | {split_audit.get('interseccion_factura_id', '—')} |

La selección de features excluyó únicamente la variable objetivo, identificadores, fechas reales y variables de leakage explícito. No se excluyeron variables por contener la palabra `corte`. Por tanto, variables derivadas válidas como `dias_transcurridos_corte` y `esta_vencida_al_corte` se conservaron cuando existían o pudieron construirse con información disponible al corte.

### 2.3 Preprocesamiento

El pipeline usa imputación por mediana para variables numéricas y `OneHotEncoder(handle_unknown="ignore")` para variables categóricas nominales. Esta corrección reemplaza el uso de `OrdinalEncoder` en variables nominales, evitando inducir un orden artificial en categorías como `ultimo_resultado_enc` o `sector_enc`.

---

## 3. Metodología de Simulación

### 3.1 Baseline limpio

El baseline corresponde a la evaluación del pipeline entrenado sobre el conjunto de prueba limpio. La degradación porcentual de cada escenario se calculó como:

\[
\text{{degradación (\%)}} = \frac{{\text{{F1-macro baseline}} - \text{{F1-macro escenario}}}}{{\text{{F1-macro baseline}}}} \times 100
\]

### 3.2 Ruido, missing y outliers

Se simularon perturbaciones de ruido gaussiano, missing aleatorio, missing dirigido por grupo de variables, outliers sobre monto y un perfil crítico combinado. El missing dirigido permite representar fallos operativos reales, por ejemplo, caída del CRM o ausencia sistemática de registros de gestión.

### 3.3 Drift temporal sin leakage

El drift temporal se corrigió para separar pasado y futuro a nivel de `factura_id`. La factura completa se asigna a train temporal o a test temporal, y el notebook valida explícitamente que la intersección de `factura_id` sea cero. Esto evita leakage entre cortes de una misma factura.

### 3.4 Drift por sector

Cuando el dataset contiene sectores one-hot encoded, el notebook evalúa todos los campos `sector_*` por separado. Para cada sector se exporta una tabla con sector, número de registros, F1-macro, recall de `+90` y degradación.

### 3.5 Pruebas de estrés

La sección de inputs fuera de rango evalúa estabilidad técnica del pipeline y necesidad de validadores antes de inferencia. No debe interpretarse como desempeño predictivo real, porque se modifican variables hacia valores imposibles en el dominio, como tasas mayores a 1, montos extremos o días negativos.

---

## 4. Resultados Baseline

| Métrica | Valor |
|---|---:|
| Accuracy | {fmt4(metricas_baseline.get('accuracy', np.nan))} |
| Balanced Accuracy | {fmt4(metricas_baseline.get('balanced_accuracy', np.nan))} |
| F1-macro | {fmt4(metricas_baseline.get('f1_macro', np.nan))} |
| F1-weighted | {fmt4(metricas_baseline.get('f1_weighted', np.nan))} |
| Precision macro | {fmt4(metricas_baseline.get('precision_macro', np.nan))} |
| Recall macro | {fmt4(metricas_baseline.get('recall_macro', np.nan))} |
| Recall +90 | {fmt4(metricas_baseline.get('recall_+90', np.nan))} |
| F1 +90 | {fmt4(metricas_baseline.get('f1_+90', np.nan))} |
| Latencia ms/reg | {fmt4(metricas_baseline.get('latencia_ms_por_reg', np.nan))} |

---

## 5. Resultados por Escenario

| Escenario | F1-macro | Balanced Accuracy | Recall +90 | Degradación F1-macro |
|---|---:|---:|---:|---:|
{lineas_escenarios}

### 5.1 Peor escenario identificado

El peor escenario de la ejecución fue **`{peor_nombre}`**, con F1-macro **{peor_f1}** y degradación **{peor_deg}** frente al baseline. En particular, el escenario solicitado **`missing_dirigido_gestion`** obtuvo F1-macro **{gestion_f1}** y degradación cercana a **{gestion_deg}**. Este resultado confirma que la pérdida simultánea de variables de gestión afecta severamente la capacidad del modelo para separar las clases de mora.

La interpretación de negocio es directa: cuando se pierden variables como `dias_desde_ultima_gestion`, `ultimo_resultado_enc`, `tasa_contacto_cliente` y `num_gestiones_factura`, el modelo deja de observar señales recientes de interacción con el cliente. Por eso cae el F1-macro hasta **{gestion_f1}**. Aunque el recall de `+90` observado en este escenario fue **{gestion_recall90}**, el desempeño global se deteriora porque el modelo pierde discriminación entre las demás clases, especialmente en fronteras intermedias como `+30` y `+60`.

---

## 6. Drift por Sector

| Sector | n_registros | F1-macro | Recall +90 | Degradación F1-macro |
|---|---:|---:|---:|---:|
{lineas_sector}

El análisis por sector permite identificar si el modelo se comporta de forma desigual ante subconjuntos de la cartera. La salida `drift_sector_results.csv` debe revisarse junto con el tamaño de muestra, porque sectores con pocos registros pueden mostrar alta variabilidad.

---

## 7. Latencia y Pruebas de Volumen

| Registros | Latencia Total (s) | Latencia/Reg (ms) |
|---:|---:|---:|
{lineas_latencia}

Las pruebas de volumen evalúan escalabilidad operativa del pipeline. Si la latencia por registro se mantiene estable al aumentar el tamaño del lote, el modelo es viable para procesamiento batch de cartera. Para inferencia transaccional, conviene monitorear la latencia en tamaños de lote pequeños.

---

## 8. Matriz de Riesgo

| Escenario | Severidad | Métrica afectada | Acción de mitigación |
|---|---|---|---|
{lineas_riesgo}

---

## 9. Análisis Crítico

El modelo muestra robustez moderada ante ruido gaussiano y outliers aislados, lo cual es esperable en modelos de árboles y boosting porque las particiones por umbral suelen tolerar pequeñas perturbaciones. Sin embargo, la robustez disminuye ante missing dirigido, especialmente cuando se anula el bloque de gestión.

El drift temporal corregido por `factura_id` permite una lectura más confiable que un split por filas. Si se mezclaran cortes de una misma factura entre train y test, las métricas podrían inflarse artificialmente. La validación de intersección nula de `factura_id` es, por tanto, una condición metodológica indispensable.

El drift por sector aporta una visión de estabilidad por grupos de negocio. Si algún sector presenta una degradación superior al promedio o bajo recall en `+90`, la recomendación es revisar calibración por segmento, enriquecer datos de ese sector o establecer reglas de revisión humana.

La prueba de inputs fuera de rango no debe usarse para afirmar que el modelo predice bien o mal bajo esos valores. Su objetivo es evidenciar si el pipeline colapsa y justificar validadores de dominio antes de inferencia: tasas en [0,1], montos positivos, días no negativos y categorías conocidas o manejadas por `OneHotEncoder(handle_unknown="ignore")`.

---

## 10. Recomendaciones

1. Mantener particiones por `factura_id` en toda evaluación futura.
2. Implementar validadores de entrada antes de inferencia para evitar tasas imposibles, montos extremos no auditados o días negativos.
3. Monitorear drift mensual con PSI/KS y desempeño por sector.
4. Revisar manualmente facturas con alto riesgo y baja confianza del modelo.
5. Reentrenar el modelo si F1-macro cae más de 10% en una ventana reciente o si PSI supera umbrales operativos.
6. Mantener `OneHotEncoder(handle_unknown="ignore")` para variables nominales y evitar pseudo-ordinalidad.

---

## 11. Artefactos Exportados

Todos los artefactos quedan organizados en `critical_scenarios_outputs`:

- `tables/baseline_metrics.csv`
- `tables/classification_report_baseline.csv`
- `tables/classification_report_worst_scenario.csv`
- `tables/noise_missing_outlier_results.csv`
- `tables/drift_results.csv`
- `tables/drift_sector_results.csv`
- `tables/temporal_drift_split_summary.csv`
- `tables/stress_test_results.csv`
- `tables/all_scenarios_summary.csv`
- `figures/confusion_matrix_baseline.png`
- `figures/confusion_matrix_worst_scenario.png`
- `report/informe_workshop_simulacion_escenarios_criticos.md`
- `artifacts/model_baseline.joblib`
- `artifacts/preprocessing_pipeline.joblib`
- `artifacts/scenario_config.json`

---

## 12. Referencias APA

Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785

Gama, J., Žliobaitė, I., Bifet, A., Pechenizkiy, M., & Bouchachia, A. (2014). A survey on concept drift adaptation. *ACM Computing Surveys, 46*(4), 1–37. https://doi.org/10.1145/2523813

He, H., & Garcia, E. A. (2009). Learning from imbalanced data. *IEEE Transactions on Knowledge and Data Engineering, 21*(9), 1263–1284. https://doi.org/10.1109/TKDE.2008.239

Moreno-Torres, J. G., Raeder, T., Alaiz-Rodríguez, R., Chawla, N. V., & Herrera, F. (2012). A unifying view on dataset shift in classification. *Pattern Recognition, 45*(1), 521–530. https://doi.org/10.1016/j.patcog.2011.06.019

Schafer, J. L., & Graham, J. W. (2002). Missing data: Our view of the state of the art. *Psychological Methods, 7*(2), 147–177. https://doi.org/10.1037/1082-989X.7.2.147

Sculley, D., Holt, G., Golovin, D., Davydov, E., Phillips, T., Ebner, D., Chaudhary, V., Young, M., Crespo, J. F., & Dennison, D. (2015). Hidden technical debt in machine learning systems. *Advances in Neural Information Processing Systems, 28*, 2503–2511.

---

*Informe generado automáticamente por el pipeline de evaluación de robustez. No contiene textos pendientes de completar.*
"""
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(informe)
    print(f"💾 Informe guardado en: {output_path}")
    return informe

informe_texto = generar_informe(
    metricas_baseline, df_all, df_riesgo, df_latencia,
    worst_global, le,
    REPORT_DIR / "informe_workshop_simulacion_escenarios_criticos.md"
)


In [ ]:
# ── CELDA 24 — Resumen ejecutivo final ────────────────────────────────────────
print("=" * 65)
print("  RESUMEN EJECUTIVO FINAL")
print("=" * 65)

baseline_f1 = metricas_baseline['f1_macro']
worst_f1    = worst_global.get('f1_macro', np.nan)
deg_worst   = calcular_degradacion(metricas_baseline, worst_global, "f1_macro")

print(f"""
╔══════════════════════════════════════════════════════════════╗
║          RESUMEN EJECUTIVO — WORKSHOP ESCENARIOS CRÍTICOS    ║
╠══════════════════════════════════════════════════════════════╣
║  Modelo: XGBoost Multiclase                                  ║
║  Dataset: {ruta_dataset.name[:45]:<45}  ║
║  Clases: {', '.join(le.classes_):<52}  ║
╠══════════════════════════════════════════════════════════════╣
║  BASELINE                                                    ║
║    F1-macro:          {baseline_f1:.4f}                                ║
║    Balanced Accuracy: {metricas_baseline.get('balanced_accuracy', 0):.4f}                                ║
║    Recall +90:        {metricas_baseline.get('recall_+90', metricas_baseline.get(f'recall_{le.classes_[-1]}', '—'))!s:<10}                               ║
╠══════════════════════════════════════════════════════════════╣
║  PEOR ESCENARIO                                              ║
║    Nombre:     {worst_global['escenario'][:47]:<47}║
║    F1-macro:   {worst_f1:.4f}                                          ║
║    Degradación:{deg_worst:.1f}%                                         ║
╠══════════════════════════════════════════════════════════════╣
║  RIESGOS CRÍTICOS IDENTIFICADOS                              ║
║    - Missing dirigido en variables de gestión                ║
║    - Drift temporal en ciclos económicos                     ║
║    - Inputs fuera de rango (fallo de integración)            ║
║    - Concentración en cartera crítica (drift covariate)      ║
╠══════════════════════════════════════════════════════════════╣
║  TOP 3 ACCIONES DE MITIGACIÓN                                ║
║    1. Validadores de rango antes de inferencia               ║
║    2. Monitoreo mensual de PSI; retraining si F1 cae >10%    ║
║    3. Revisión humana para facturas con riesgo extremo       ║
╚══════════════════════════════════════════════════════════════╝
""")

# Verificar archivos generados
print("\n📁 ARCHIVOS GENERADOS:")
for subdir in [TABLES_DIR, FIGURES_DIR, REPORT_DIR, ARTIFACTS_DIR]:
    print(f"\n  📂 {subdir.name}/")
    for f in sorted(subdir.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"     {'✅'} {f.name}  [{size_kb:.1f} KB]")

In [ ]:
# ── CELDA 25 — Comprimir outputs para descarga ─────────────────────────────────
import shutil

zip_path = "/content/workshop_escenarios_criticos_outputs.zip"
shutil.make_archive(
    zip_path.replace(".zip", ""),
    "zip",
    "/content",
    "critical_scenarios_outputs"
)
print(f"📦 ZIP generado: {zip_path}")
print(f"   Tamaño: {Path(zip_path).stat().st_size / 1024:.1f} KB")
print("\n⬇️  Descarga disponible desde el panel de archivos de Colab.")
print("   Ruta: /content/workshop_escenarios_criticos_outputs.zip")